In [ ]:
year = '2023/2024'  # input year in the format year = '2018/2019'
version = '1.3'  # version of the register file
bulletinversion = '0.0.1'  # version of the statsbulletin file - CHANGE VERSION TO BULLETINVERSION IN CODE
total_government_expenditure = 1059.51  # insert total government expenditure
signoff_date = '30/01/2026'  # this is the deadline date you give to the dept to get the data signed off by
#Department_to_be_exported = 'CO'  #E.g. BEIS, comment line out if wish to run all department and comment out line this variable is used LINK TO VARIABLE

In [ ]:
# !pip3 install simple_salesforce
# !pip3 install python-dotenv
# !pip install python-docx
# !pip install SalesforceAuthenticationFailed
# !pip install odfpy

In [ ]:
!pip3 install xlrd
!pip3 install PyPDF2

In [ ]:
!pip3 install xlsxwriter

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import string
import xlsxwriter  #to write dataframes to excel
import statistics
from datetime import datetime
import os
from pathlib import Path
from win32com.client import Dispatch
import statistics
#from textdistance import Levenshtein
import xlwings as xw
import xlrd
import PyPDF2
import docx
from PIL import Image, ImageDraw, ImageFont
from simple_salesforce import Salesforce
from dotenv import load_dotenv
import requests
from io import StringIO, BytesIO
from matplotlib import rc
from os.path import expanduser
home = expanduser("~")
import glob

load_dotenv('GGISCredentials.env')  #make sure that you have edited this with your own credentials

In [ ]:
# =============================================================
# Encoding utilities — single source of truth
# Pipeline: Salesforce/API -> UTF-8 -> pandas -> UTF-8 CSV -> pandas -> XLSX
# =============================================================
import re
import unicodedata
import pandas as pd

CSV_IN_ENCODINGS = ("utf-8-sig", "utf-8", "cp1252")
CSV_OUT_ENCODING = "utf-8-sig"  # BOM helps Excel recognise UTF-8 CSVs

# Common markers produced when UTF-8 bytes are decoded as Windows-1252/Latin-1.
_MOJIBAKE_MARKERS = ("Ã", "Â", "â€", "â€™", "â€œ", "â€", "â€“", "â€”", "ï»¿", "�")

def fix_mojibake(value):
    """Repair common UTF-8-as-cp1252 mojibake without deleting valid characters."""
    if not isinstance(value, str):
        return value

    value = unicodedata.normalize("NFC", value).replace("\ufeff", "")
    out = value

    # Reverse a bad cp1252 decode. Two passes also repairs double-encoded text.
    for _ in range(2):
        if not any(marker in out for marker in _MOJIBAKE_MARKERS[:-1]):
            break
        try:
            candidate = out.encode("cp1252", errors="strict").decode("utf-8", errors="strict")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break
        if candidate == out:
            break
        out = candidate

    return unicodedata.normalize("NFC", out)

def clean_frame(df):
    """Normalise column names and all string cells in a dataframe."""
    df = df.copy()
    df = df.rename(columns=lambda c: fix_mojibake(c) if isinstance(c, str) else c)
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].map(fix_mojibake)
    return df

def read_csv_clean(source, **kwargs):
    """Read CSV as UTF-8 first; use cp1252 only for genuine legacy files."""
    sep = kwargs.pop("sep", ",")
    last_error = None
    for encoding in CSV_IN_ENCODINGS:
        try:
            if hasattr(source, "seek"):
                source.seek(0)
            df = pd.read_csv(source, sep=sep, encoding=encoding, **kwargs)
            return clean_frame(df)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

def find_mojibake(df):
    """Return suspect cells so bad text cannot silently reach sign-off workbooks."""
    issues = []
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        for idx, value in df[col].items():
            if not isinstance(value, str):
                continue
            markers = [m for m in _MOJIBAKE_MARKERS if m in value]
            if markers:
                issues.append({
                    "row": idx,
                    "column": col,
                    "markers": ", ".join(markers),
                    "value": value,
                })
    return pd.DataFrame(issues)


In [ ]:
DATA_SOURCE = "api"  # 'files' or 'api'
FORCE_DATE = None
DATA_DIR = '../data'

FILE_PATTERNS = {
    'scheme': '*GGIS scheme download for register.csv',
    'awards': '*GGIS award download for register.csv',
    'orgs': '*GGIS orgs download for register.csv',
    'acronyms': '*-DepartmentAcronyms.csv',
    'general_top10': '*General_Top10.csv',
    'formula_top10': '*Formula_Top10.csv',
    'scheme_purposes': '*SchemePurposes.csv',
}

#Publication
YEAR = '2023/2024'
VERSION = '1.2'
BULLETIN_VERSION = '0.0.1'
TOTAL_GOVERNMENT_EXPENDITURE = 1229.936
SIGNOFF_DATE = '31/01/2026'

load_dotenv('GGISCredentials.env')  # make sure that you have edited this with your own credentials

In [ ]:
def find_latest_file(pattern, date_override=None):
    """Find recent files using glob"""
    search_path = os.path.join(DATA_DIR, pattern)
    files = glob.glob(search_path, recursive=False)

    if not files:
        raise FileNotFoundError(f"No files found matching: {search_path}")

    if date_override:
        dated_pattern = f"{date_override}-{pattern.split('-')[-1]}"
        dated_files = [
            f for f in files
            if os.path.basename(f).startswith(date_override)
        ]
        if dated_files:
            return dated_files[0]

        raise FileNotFoundError(
            f"No files found for data {date_override} matching: {pattern}"
        )

    # recently modified file
    return max(files, key=os.path.getmtime)


def load_data_from_files():
    # Load files local csv
    try:
        print("Loading data from files...")

        today = FORCE_DATE if FORCE_DATE else datetime.today().strftime('%Y-%m-%d')

        # Load core datasets
        data_files = {
            'department_acronym': find_latest_file(
                FILE_PATTERNS['acronyms'], FORCE_DATE
            ),
            'scheme_df': find_latest_file(
                FILE_PATTERNS['scheme'], FORCE_DATE
            ),
            # NOTE: award_df used to be a second, identical read of the same
            # file. It is now derived from awards_df below (see return block).
            'awards_df': find_latest_file(
                FILE_PATTERNS['awards'], FORCE_DATE
            ),
            'orgs_df': find_latest_file(
                FILE_PATTERNS['orgs'], FORCE_DATE
            ),
            'general_df': find_latest_file(
                FILE_PATTERNS['general_top10'], FORCE_DATE
            ),
            'formula_df': find_latest_file(
                FILE_PATTERNS['formula_top10'], FORCE_DATE
            ),
            'scheme_purposes': find_latest_file(
                FILE_PATTERNS['scheme_purposes'], FORCE_DATE
            ),
        }

        # Loading confirmation
        print("\nLoaded data files:")
        for name, path in data_files.items():
            print(f"- {name}: {os.path.basename(path)}")

        # Read all csv
        loaded = {
            k: read_csv_clean(v, dtype="object")
            for k, v in data_files.items()
        }
        # award_df is the same source data as awards_df; keep an independent
        # copy so the two downstream pipelines cannot mutate each other.
        loaded['award_df'] = loaded['awards_df'].copy()
        return loaded

    except Exception as e:
        print(f"\nError loading files: {str(e)}")
        raise


def load_data_from_api():
    """load by salesforce api"""
    try:
        print("Loading data from API...")
        load_dotenv('GGISCredentials.env')

        sf = Salesforce(
            username=os.getenv('ggisemail'),
            password=os.getenv('password'),
            security_token=os.getenv('security_token'),
            domain='login'
        )

        # Import URL and Export parameters from .env file
        sf_org = os.getenv("sf_org")
        export_params = '?isdtp=p1&export=1&enc=UTF-8&xf=csv'

        # Import report IDs from .env
        SchemeData_Report_ID = os.getenv("SchemeData_Report_ID")
        AwardData_Report_ID = os.getenv("AwardData_Report_ID")
        OrgsWithAwards_Report_ID = os.getenv("OrgswithAwards_Report_ID")
        Formula_Report_ID = os.getenv("Formula_Report_ID")
        General_Report_ID = os.getenv("General_Report_ID")
        SchemePurpose_Report_ID = os.getenv("SchemePurpose_Report_ID")

        # Download Scheme Data Report
        sfUrl = sf_org + SchemeData_Report_ID + export_params
        response = requests.get(sfUrl, headers=sf.headers, cookies={'sid': sf.session_id})
        response.raise_for_status()
        scheme_df = read_csv_clean(BytesIO(response.content), dtype="object")

        # Download Award Data Report
        sfUrl = sf_org + AwardData_Report_ID + export_params
        response = requests.get(sfUrl, headers=sf.headers, cookies={'sid': sf.session_id})
        response.raise_for_status()
        awards_df = read_csv_clean(BytesIO(response.content), dtype="object")
        # award_df was a second identical parse of the same payload; a copy of
        # the parsed frame is equivalent and much cheaper.
        award_df = awards_df.copy()

        # Download Orgs with Award Data Report
        sfUrl = sf_org + OrgsWithAwards_Report_ID + export_params
        response = requests.get(sfUrl, headers=sf.headers, cookies={'sid': sf.session_id})
        response.raise_for_status()
        orgs_df = read_csv_clean(BytesIO(response.content), dtype="object")

        # Download Formula Data Report
        sfUrl = sf_org + Formula_Report_ID + export_params
        response = requests.get(sfUrl, headers=sf.headers, cookies={'sid': sf.session_id})
        response.raise_for_status()
        formula_df = read_csv_clean(BytesIO(response.content), dtype="object")

        # Download General Data Report
        sfUrl = sf_org + General_Report_ID + export_params
        response = requests.get(sfUrl, headers=sf.headers, cookies={'sid': sf.session_id})
        response.raise_for_status()
        general_df = read_csv_clean(BytesIO(response.content), dtype="object")

        # Download Scheme Purpose Report
        sfUrl = sf_org + SchemePurpose_Report_ID + export_params
        response = requests.get(sfUrl, headers=sf.headers, cookies={'sid': sf.session_id})
        response.raise_for_status()
        scheme_purposes = read_csv_clean(BytesIO(response.content), dtype="object")

        # Loading supporting files
        # general_df / formula_df are the API-sourced top-10 reports above, so
        # only the acronym lookup still needs to come from a local file.
        data_files = {
            'department_acronym': find_latest_file(FILE_PATTERNS['acronyms']),
        }

        return {
            'scheme_df': scheme_df,
            'awards_df': awards_df,
            'award_df': award_df,
            'orgs_df': orgs_df,
            'general_df': general_df,
            'formula_df': formula_df,
            'scheme_purposes': scheme_purposes,
            **{
                k: read_csv_clean(v)
                for k, v in data_files.items()
            }
        }

    except Exception as e:
        print(f"API Error: {str(e)}")
        raise        

In [ ]:
os.getcwd()

In [ ]:
path = os.getcwd()

In [ ]:
x = path + "\\Scripts\\data"

In [ ]:
os.chdir(x)

In [ ]:
if __name__ == "__main__":
    try:
        # Load data based on setting
        if DATA_SOURCE == 'files':
            data = load_data_from_files()
        elif DATA_SOURCE == 'api':
            data = load_data_from_api()
        else:
            raise ValueError("DATA_SOURCE must be 'files' or 'api'")

        # unpack data variables
        department_acronym = data['department_acronym']
        scheme_df = data['scheme_df']
        awards_df = data['awards_df']
        award_df = data['award_df']
        orgs_df = data['orgs_df']
        general_df = data['general_df']
        formula_df = data['formula_df']
        scheme_purposes = data['scheme_purposes']

        print("\nStarting Data processing...")

#        awards_df = awards_df[awards_df['Awards status'] != "Cancelled"]
#        scheme_df = scheme_df.rename(columns={"FY value (budgeted or actual)": "FY value"})

        print("\nProcessing completed successfully!")
    except Exception as e:
        print(f"\nError in processing: {str(e)}")
        raise

In [ ]:
scheme_purposes.columns

In [ ]:
#Remove schemes/awards that spent in the year prior but not in the specified year, or otherwise need to be manually removed:
####Schemes
#DBT:
scheme_df = scheme_df[scheme_df['Scheme Reference #'] != "G2-SCH-2021-01-12345"]

####Awards
#CO:
awards_df = awards_df[awards_df['Award Reference #'] != "G2-GA-2021010012345"]


In [ ]:
#Remove any cancelled awards
awards_df = awards_df[awards_df['Award status'] != "Cancelled"]

In [ ]:
#Make changes to award and scheme FY value columns to take into account both budgeted and actual values
#for scheme data, GGIS combined column is already calculated
scheme_df = scheme_df.rename(columns={"FY value (budgeted or actual)": "FY value"})

#replace nulls with 0 in amount paid column
awards_df['Amount paid'] = awards_df['Amount paid'].fillna(0)

#currently not picking up 0s in below:

#for award data, need to create combined column within script:
awards_df['Amount Awarded'] = np.where((awards_df['Amount paid'] == 0) | (awards_df['Amount paid'].isnull()) | (awards_df['Amount paid'] == '0'),
                                      np.where((awards_df['Amount paid'] == '0.00'),
                                               awards_df['Budgeted Agreement value FY'],
                                               awards_df['Amount paid']))

#renaming original columns for purpose of updating script (to make sure old columns aren't used unknowingly)
awards_df = awards_df.rename(columns={'Budgeted Agreement value FY': 'Budgeted Agreement value FY (Original)', 'Amount paid':'Amount paid (Original)'})

In [ ]:
# Date variables automated
today = datetime.today().strftime('%Y-%m-%d')
yy = year.replace("/", " to ")
yyy = yy
yyyy = year[2:4] + '/' + year[-2:]
y = year[2:4] + year[-2:]
y1 = year[0:4]
y2 = year[5:9]
start_of_fy = "01/04/" + y1
year_suffix = "-" + year[2:4] + "-" + year[-2:]
print(yyyy)

In [ ]:
# Text is now normalised at load time by read_csv_clean()/clean_frame().
#
# The previous approach chained blind replacements:
#     df.replace('Ã£','£').replace('Â','').replace('â','')
# which was the main source of garbled sign-off files:
#   * 'Ã£' is actually 'ã', not '£' - it corrupted Portuguese/Welsh names.
#   * deleting every 'Â' and 'â' also deleted valid characters in recipient
#     names and destroyed the second byte of '£', '–', ''' sequences,
#     leaving half-repaired text that can never be recovered.
#
# clean_frame() reverses the faulty decode instead, which is lossless.
for _name in [
    'awards_df', 'scheme_df', 'scheme_purposes', 'award_df',
    'orgs_df', 'general_df', 'formula_df',
]:
    if _name in globals():
        globals()[_name] = clean_frame(globals()[_name])


In [ ]:
#Applying filters and conditions to scheme_df
print(len(scheme_df))
scheme_data = scheme_df[scheme_df['Scheme FY breakdown Name'].astype(str).str.contains(yyyy)].reset_index(drop=True)
print(len(scheme_data))
scheme_data = scheme_data.drop_duplicates(subset=['Scheme Reference #'],keep="first").reset_index(drop=True)
print(len(scheme_data))
scheme_data = scheme_data[scheme_data['Public funding source']=="UK Exchequer"].reset_index()
print(len(scheme_data))

In [ ]:
scheme_data["acronym_merge"] = scheme_data["Funder: Department abbreviation"].str.lower()
department_acronym['acronym_merge'] = department_acronym['Acronym'].str.lower()

print(scheme_data["acronym_merge"].value_counts())

scheme_data = pd.merge(
    scheme_data,
    department_acronym,
    how = 'left',
    left_on = 'acronym_merge',
    right_on = 'acronym_merge'
)
#print(len(awards_df))

print(scheme_data["acronym_merge"].value_counts())

In [ ]:
award_orgs = pd.merge(
    awards_df,
    orgs_df,
    how = 'left',
    on = "Award reference #"
)
#print(len(award_orgs))

award_orgs = pd.merge(
    award_orgs,
    scheme_data,
    how = 'left',
    left_on = "Grant Scheme: Scheme Reference #",
    right_on = "Scheme Reference #"
)
#print(len(award_orgs))

#print(award_orgs["acronym_merge"].value_counts())

award_orgs = award_orgs[award_orgs['Scheme name'].notnull()]
#print(len(award_orgs))

#print(award_orgs["acronym_merge"].value_counts())

In [ ]:
department_acronym['Funder: Organisation Name'] = department_acronym['Funder: Organisation Name'].replace(
    {'Department for Digital, culture, media, & sport': 'Department for Culture, Media and Sport'}
)

In [ ]:
folder = f"../data/" + today + '_Government_Grants_Register_exports'
if not os.path.exists(folder):
    os.makedirs(folder)

In [ ]:
path0 = folder+'/' +today + '_Government_Grants_Register_' + yy + '.xlsx'    #name of excel that will be exported
writer = pd.ExcelWriter(path0, engine = 'xlsxwriter') #openning function that writes data to the open excel

In [ ]:
#create dummy tabs so that order is correct:
dummy = pd.DataFrame()

# Here go the dummy sheets in the order you want
dummy.to_excel(writer, sheet_name='Contents')
dummy.to_excel(writer, sheet_name='Schemes')
dummy.to_excel(writer, sheet_name='Awards')

In [ ]:
department_acronym['Funder: Organisation Name'] = department_acronym['Funder: Organisation Name'].replace(
    {'Department for Digital, culture, media, & sport': 'Department for Culture, Media and Sport'}
)

In [ ]:
folder = f"../data/" + today + '_Government_Grants_Register_exports'
if not os.path.exists(folder):
    os.makedirs(folder)

In [ ]:
path0 = folder+'/' +today + '_Government_Grants_Register_' + yy + '.xlsx'  #name of excel that will be exported
writer = pd.ExcelWriter(path0, engine = 'xlsxwriter')  #openning function that writes data to the open excel

In [ ]:
#create dummy tabs so that order is correct:
dummy = pd.DataFrame()

# Here go the dummy sheets in the order you want
dummy.to_excel(writer, sheet_name='Contents')
dummy.to_excel(writer, sheet_name='Schemes')
dummy.to_excel(writer, sheet_name='Awards')
#dummy.to_excel(writer, sheet_name='Scheme_field_notes')
#dummy.to_excel(writer, sheet_name='Award_field_notes')
dummy.to_excel(writer, sheet_name='Meta')

#This method didn't work with the scheme/award field notes as that uses a different method to populate the tabs
#for now these tabs will need to be reordered manually after producing the file

In [ ]:
#Draft name of excel file
mydate = datetime.now()
mydate.strftime("%B") + ' ' + mydate.strftime("%Y")

In [ ]:
#Creating first tab - 'Contents'
workbook = writer.book
###worksheet = workbook.add_worksheet('#Contents')
worksheet = workbook.get_worksheet_by_name('#Contents')

#Cell-by-Cell
# Add A1
A1 = workbook.add_format({'bold': True, 'font_size':24})
worksheet.write('A1', f"Government grants register {yy} - scheme and award data", A1)

#Add A2
A2 = workbook.add_format({'italic': True, 'font_size':11})
worksheet.write('A2', f"Data tables - {mydate.strftime('%B')} " + " " + mydate.strftime("%Y"), A2)

#Add A4
A4 = workbook.add_format({'bold': True, 'font_size':14})
worksheet.write('A4', 'Introduction', A4)

#Add A5 and A6
worksheet.write(
    'A5',
    f"This document contains the Government grants register {yyy} - providing details on general and formula grants at scheme and award level."
)

#Add A8
worksheet.write('A8', 'Contents', A4)

#Add A9
A10 = workbook.add_format({'bold': True, 'italic': True, 'font_size':11})
worksheet.write('A9', 'Tab', A10)

#Add B10
worksheet.write('B9', 'Description', A10)
#hyperlinks to other sheets done after those sheets have been created

In [ ]:
scheme_data['sort_on'] = [x.strip()[-6:] for x in scheme_data['Scheme Reference #']]
scheme_data['Do not publish'].value_counts()

In [ ]:
# Create scheme dataframe
fixed_scheme_data = scheme_data[scheme_data['Do not publish'] == "0"].reset_index(drop=True)
print(len(fixed_scheme_data))

fixed_scheme_data['sort_on'] = [x.strip()[-6:] for x in fixed_scheme_data['Scheme Reference #']]

In [ ]:
# Create scheme dataframe for the relevant year
fixed_scheme_data = fixed_scheme_data.rename(columns={
    "Funder": "Funding Org Name",
    "Scheme Reference #": "Grant Programme:Code",
    "Scheme Name": "Grant Programme:Title",
    "Scheme aims and objectives": "Description",
    "Scheme Value per year": "Value per year",
    "Budgeted or Actual? (FY value)": "Budgeted or Actual? (FY value)",
    "Scheme start date": "Start Date",
    "Scheme end date": "End Date",
    "Scheme allocation method": "Allocation method",
    "Expenditure by function": "COFOG: Sector type",
    "Funder: Organisation Name_x": "Funder: Organisation Name",
    "Authority act: Authority Act Name": "Authority Act"
})

In [ ]:
fixed_scheme_data['Duration (Months)'] = pd.to_numeric(
    fixed_scheme_data['Scheme duration']
) * 12

In [ ]:
#Loop over data to create column with relevant cofog names
#convert start & end date to strings
fixed_scheme_data['Start Date'] = fixed_scheme_data['Start Date'].astype(str)
fixed_scheme_data['End Date'] = fixed_scheme_data['End Date'].astype(str)

fixed_scheme_data['Start Date'] = pd.to_datetime(
    fixed_scheme_data['Start Date'],
    format='%d/%m/%Y'
)

fixed_scheme_data['End Date'] = pd.to_datetime(
    fixed_scheme_data['End Date'],
    format='%d/%m/%Y'
)

fixed_scheme_data['Start Date'] = pd.to_datetime(
    fixed_scheme_data['Start Date'],
    format='%Y-%m-%d'
)

fixed_scheme_data['End Date'] = pd.to_datetime(
    fixed_scheme_data['End Date'],
    format='%Y-%m-%d'
)

In [ ]:
#for i in range(len(fixed_scheme_data)):
#    fixed_scheme_data.at[i,'Start Date'] = datetime.strptime(
#        fixed_scheme_data.at[i,'Start Date'], '%d/%m/%Y'
#    ).strftime("%Y-%m-%d")
#    fixed_scheme_data.at[i,'End Date'] = datetime.strptime(
#        fixed_scheme_data.at[i,'End Date'], '%d/%m/%Y'
#    ).strftime("%Y-%m-%d")

In [ ]:
#commenting this line to leave full allocation method in
#fixed_scheme_data['Allocation method'] = fixed_scheme_data['Allocation method'].str[:7]

In [ ]:
fixed_scheme_data = fixed_scheme_data[[
    'Funder: Organisation Name',
    'Managed by: Organisation Name',
    'Grant Programme:Code',
    'Scheme name',
    'Description',
    'FY value',
    'Start Date',
    'End Date',
    'Allocation method',
    'Duration (Months)',
    'COFOG: Sector type',
    'Authority Act: Authority Act Name',
    'sort_on'
]]

In [ ]:
schemes = fixed_scheme_data.drop_duplicates(
    subset=['Grant Programme:Code'],
    keep='first'
).reset_index()

print(len(schemes))

schemes = schemes.sort_values(by=["sort_on"]).reset_index()

In [ ]:
# Only strip the Unicode replacement character (U+FFFD), which is what an
# undecodable byte turns into. The old code deleted every real '?' from
# scheme descriptions, which is why sign-off text lost its punctuation.
why = 0
for i in range(len(schemes)):
    if '\ufffd' in str(schemes.at[i,'Description']):
        schemes.at[i,'Description'] = schemes.at[i,'Description'].replace('\ufffd', "")
        why += 1

print(why)

In [ ]:
for i in range(len(schemes)):
    if '\ufffd' in str(schemes.at[i,'Scheme name']):
        schemes.at[i,'Scheme name'] = schemes.at[i,'Scheme name'].replace('\ufffd', "")
        why += 1

print(why)

why = 0

In [ ]:
why = 0
#test for ?
for i in range(len(schemes)):
    if '?' in str(schemes.at[i,'Description']):
        why += 1

print(why)

capitala = 0
#test for ?
for i in range(len(schemes)):
    if 'Â' in str(schemes.at[i,'Description']):
        capitala += 1

print(capitala)

In [ ]:
scheme = schemes[[
    'Funder: Organisation Name',
    'Managed by: Organisation Name',
    'Grant Programme:Code',
    'Scheme name',
    'Description',
    'FY value',
    'Start Date',
    'End Date',
    'Allocation method',
    'Duration (Months)',
    'COFOG: Sector type',
    'Authority Act: Authority Act Name'
]]

scheme = scheme.rename(columns={"Scheme name":"Grant Programme:Title"})

In [ ]:
#Convert fields to correct format, so they will be in correct format in excel:
#Date fields (as dates)
scheme['Start Date'] = pd.to_datetime(
    scheme['Start Date'],
    format='%Y-%m-%d'
).dt.date

scheme['End Date'] = pd.to_datetime(
    scheme['End Date'],
    format='%Y-%m-%d'
).dt.date

#Number fields (as numbers)
scheme['FY value'] = pd.to_numeric(scheme['FY value'])

In [ ]:
#Writing scheme to excel sheet
scheme.to_excel(writer, sheet_name='#Schemes', index=False)

In [ ]:
#amending column formats for numbers and dates

# Get the xlsxwriter workbook and worksheet objects.
workbook = writer.book
worksheet = writer.sheets['#Schemes']

# Add some cell formats.
num_format = workbook.add_format({'num_format': '0.00'})
date_format = workbook.add_format({'num_format': 'yyyy-mm-dd'})

# Note: It isn't possible to format any cells that already have a format such
# as the index or headers or any cells that already contain dates or datetimes.

# Set the format (5th column is the FY value)
worksheet.set_column(5, 5, None, num_format)

# Set the format (7th/8th columns is the start/end date)
worksheet.set_column(7, 8, None, date_format)

In [ ]:
pd.set_option("display.max_rows", None, "display.max_columns", None)
award_orgs.columns

In [ ]:
data = award_orgs

data['Companies House Number'] = data['Companies House Number'].fillna(0)
data['Charity Commission registered number'] = data['Charity Commission registered number'].fillna(0)
data['GB-UKPRN'] = data['GB-UKPRN'].fillna(0)

In [ ]:
def get_identifier(row):
    if(row['Recipient Org:Company Number'] != "" and any(i.isdigit() for i in str(row['Recipient Org:Company Number']))):
        chn = str(row['Recipient Org:Company Number'])
        if(len(chn) <= 8):

In [ ]:
def get_identifier(row):
    if(row['Recipient Org:Company Number'] != "" and any(i.isdigit() for i in str(row['Recipient Org:Company Number']))):
        chn = str(row['Recipient Org:Company Number'])
        if(len(chn) <= 8):
            extra_zeros = (8 - len(chn)) * "0"
            chn = extra_zeros + chn
            identifier = "GB-COH-" + chn
        else:
            identifier = "360G-cabinetoffice-" + str(row['18 Character ID'])
    elif(row['Recipient Org:Charity Number'] != "" and any(i.isdigit() for i in str(row['Recipient Org:Charity Number']))):
        identifier = "GB-CHC-" + str(row['Recipient Org:Charity Number'])
    elif(row['GB-UKPRN'] != "" and row['GB-UKPRN'] != 0 and row['GB-UKPRN'] != "0" and any(i.isdigit() for i in str(row['GB-UKPRN']))):
        identifier = "GB-UKPRN-" + str(int(float(row['GB-UKPRN'])))
    else:
        identifier = "360G-cabinetoffice-" + str(row['18 Character ID'])

    return identifier

#Will note - I have changed the above to use the '18 Character ID' for the organisation, rather than the 'Organisation ID'
#The 18 character ID is the same as the org ID but with 3 extra letters at the end
#The org ID is unique only if you are looking at it case sensitive, otherwise there are dupes
#The 18 character ID seems to be unique completely, regardless of case sensitivity

In [ ]:
def correct_CHN(row):
    if(row['Recipient Org:Company Number'] != "" and any(i.isdigit() for i in str(row['Recipient Org:Company Number']))):
        chn = str(row['Recipient Org:Company Number'])
        if(len(chn) <= 8):
            extra_zeros = (8 - len(chn)) * "0"
            new_chn = extra_zeros + chn
        else:
            new_chn = ""
    else:
        new_chn = ""

    return(new_chn)

In [ ]:
# Create award dataframe
award_with_recipient = data

# Create award dataframe for the relevant year
print(len(award_with_recipient))

award_with_recipient = award_with_recipient[
    award_with_recipient['Do not publish_x'] != "1"
].reset_index(drop=True)

print(len(award_with_recipient))

award_with_recipient = award_with_recipient[
    award_with_recipient['Do not publish_y'] != "1"
].reset_index(drop=True)

print(len(award_with_recipient))

award_with_recipient = award_with_recipient[
    award_with_recipient['Awards FY breakdown Name'].str.contains(yyyy)
].reset_index(drop=True)

award_with_recipient = award_with_recipient.drop_duplicates(
    subset=["Award reference #"]
).reset_index(drop=True)

print(len(award_with_recipient))

In [ ]:
award_with_recipient = award_with_recipient.rename(columns={
    "Grant Award Name": "Title",
    "Grant Award aims and Objectives": "Description",
    "Grant Scheme: System Scheme Reference No": "Grant Programme:Code",
    "Scheme name": "Grant Programme:Title",
    "Award Start Date": "Award Date",
    "Recipient name: Organisation Name": "Recipient Org:Name",
    "Companies House Number": "Recipient Org:Company Number",
    "Charity Commission registered number": "Recipient Org:Charity Number",
    "Billing Street": "Recipient Org:Street Address",
    "Billing Zip/Postal Code": "Recipient Org:Postal Code",
    "Billing Country": "Recipient Org:Country",
    "Billing City": "Recipient Org:City",
    "Funder: Organisation Name_x": "Funding Org:Name",
    "Allocation method_x": "Allocation Method",
    "Grant Award: Last Modified Date": "Last modified",
    "Number of recipient(s)": "Number of recipients",
    "Scheme name": "Grant Programme:Title",
    "Award name": "Title",
    "Grant award aims and objectives": "Description",
    "Grant Scheme: Scheme Reference #": "Grant Programme:Code",
    "Authority Act: Authority Act Name": "Authority Act"
})

In [ ]:
#Fix teh bottom 2
award_with_recipient["Identifier"] = "360G-cabinetoffice-" + award_with_recipient["Award reference #"] + year_suffix

award_with_recipient['sort_on'] = [x.strip()[-6:] for x in award_with_recipient['Award reference #']]

In [ ]:
#award_with_recipient['Duration (Months)'] = award_with_recipient['Award Duration (Years)'] * 12
award_with_recipient['Currency'] = 'GBP'
award_with_recipient['From an open call?'] = np.where(
    award_with_recipient['Allocation Method'] == 'General grants - Competed',
    'Yes',
    'No'
)
award_with_recipient['Recipient Name: Primary Address 1'] = award_with_recipient['Recipient Org:Street Address'].fillna("")
award_with_recipient['Recipient Org:Street Address'] = award_with_recipient['Recipient Org:Street Address']
award_with_recipient['Number of recipients'] = award_with_recipient['Number of recipients'].fillna("1")
award_with_recipient['Recipient Org:Postal Code'] = award_with_recipient['Recipient Org:Postal Code'].str.replace(" ", "")
award_with_recipient['Recipient Org:Street Address'] = award_with_recipient['Recipient Org:Street Address'].str.replace("n/a, n/a", "")

In [ ]:
award_with_recipient.loc[
    award_with_recipient['Recipient Org:Company Number'] == 0,
    ['Recipient Org:Company Number']
] = ""

award_with_recipient.loc[
    award_with_recipient['Recipient Org:Charity Number'] == 0,
    ['Recipient Org:Charity Number']
] = ""

award_with_recipient.loc[
    award_with_recipient['Recipient Org:Company Number'] == "0",
    ['Recipient Org:Company Number']
] = ""

award_with_recipient.loc[
    award_with_recipient['Recipient Org:Charity Number'] == "0",
    ['Recipient Org:Charity Number']
] = ""

In [ ]:
award_with_recipient['Recipient Org:Identifier'] = award_with_recipient.apply(get_identifier, axis=1)

award_with_recipient.loc[
    award_with_recipient['Award type'] == "Individual",
    ['Recipient Org:Identifier']
] = ""

In [ ]:
#remove dodgy company numbers from company number field
award_with_recipient['Recipient Org:Company Number'] = award_with_recipient.apply(correct_CHN, axis=1)

In [ ]:
#Question marks in titles and descriptions
# Strip only undecodable characters (U+FFFD), never genuine '?' punctuation.
award_with_recipient['Description'] = (
    award_with_recipient['Description'].astype(str).str.replace('\ufffd', '', regex=False)
)
award_with_recipient['Title'] = (
    award_with_recipient['Title'].astype(str).str.replace('\ufffd', '', regex=False)
)

In [ ]:
award_with_recipient['Recipient Org:Identifier'] = np.where(
    award_with_recipient['Recipient Org:Identifier'] == "",
    award_with_recipient["Identifier"],
    award_with_recipient['Recipient Org:Identifier']
)

In [ ]:
# Report (do not silently delete) any rows that still contain undecodable
# characters after cleaning - these indicate a bad source record in GGIS.
why = int(
    award_with_recipient['Title'].astype(str).str.contains('\ufffd').sum()
    + award_with_recipient['Description'].astype(str).str.contains('\ufffd').sum()
)
award_with_recipient['Title'] = award_with_recipient['Title'].astype(str).str.replace('\ufffd', '', regex=False)
award_with_recipient['Description'] = award_with_recipient['Description'].astype(str).str.replace('\ufffd', '', regex=False)
print(why)

In [ ]:
boopboop = award_with_recipient[award_with_recipient['Recipient Org:Identifier'] == ""]
boopboop

In [ ]:
###Redactions###

#This section uses the redacted fields sheet that should have been downloaded earlier
#22/23 Google sheet is here: https://drive.google.com/file/d/1PM-rvX7vrQfARHmyHw7iYTp_e__B0kM2/view?usp=drive_link
#Save the main 'combined' sheet at data/redacted_fields_FY22-23.csv
redacted_fields = pd.read_excel(r'..\data\redacted_fields FY23-24.xlsx', 1)

award_with_recipient = award_with_recipient.merge(
    redacted_fields,
    how = "left",
    left_on = "Identifier",
    right_on = "Identifier"
)

award_with_recipient['Recipient Org:Name'] = np.where(
    award_with_recipient['Recipient Org:Name_y'].isnull(),
    award_with_recipient['Recipient Org:Name_x'],
    award_with_recipient['Recipient Org:Name_y']
)

award_with_recipient['Amount Awarded'] = np.where(
    award_with_recipient['Amount Awarded_y'].isnull(),
    award_with_recipient['Amount Awarded_x'],
    award_with_recipient['Amount Awarded_y']
)

award_with_recipient['Recipient Org:Identifier'] = np.where(
    award_with_recipient['Recipient Org:Identifier_y'].isnull(),
    award_with_recipient['Recipient Org:Identifier_x'],
    award_with_recipient['Recipient Org:Identifier_y']
)

#newly added by will (added in new cols needing amending into redaction doc)
award_with_recipient['Recipient Org:Charity Number'] = np.where(
    award_with_recipient['Recipient Org:Charity Number_y'].isnull(),
    award_with_recipient['Recipient Org:Charity Number_x'],
    award_with_recipient['Recipient Org:Charity Number_y']
)

award_with_recipient['Recipient Org:Company Number'] = np.where(
    award_with_recipient['Recipient Org:Company Number_y'].isnull(),
    award_with_recipient['Recipient Org:Company Number_x'],
    award_with_recipient['Recipient Org:Company Number_y']
)

award_with_recipient['Recipient Org:Street Address'] = np.where(
    award_with_recipient['Recipient Org:Street Address_y'].isnull(),
    award_with_recipient['Recipient Org:Street Address_x'],
    award_with_recipient['Recipient Org:Street Address_y']
)

award_with_recipient['Recipient Org:City'] = np.where(
    award_with_recipient['Recipient Org:City_y'].isnull(),
    award_with_recipient['Recipient Org:City_x'],
    award_with_recipient['Recipient Org:City_y']
)

award_with_recipient['Recipient Org:Country'] = np.where(
    award_with_recipient['Recipient Org:Country_y'].isnull(),
    award_with_recipient['Recipient Org:Country_x'],
    award_with_recipient['Recipient Org:Country_y']
)

award_with_recipient['Recipient Org:Postal Code'] = np.where(
    award_with_recipient['Recipient Org:Postal Code_y'].isnull(),
    award_with_recipient['Recipient Org:Postal Code_x'],
    award_with_recipient['Recipient Org:Postal Code_y']
)

award_with_recipient['Description'] = np.where(
    award_with_recipient['Description_y'].isnull(),
    award_with_recipient['Description_x'],
    award_with_recipient['Description_y']
)

award_with_recipient['Grant Programme:Title'] = np.where(
    award_with_recipient['Grant Programme:Title_y'].isnull(),
    award_with_recipient['Grant Programme:Title_x'],
    award_with_recipient['Grant Programme:Title_y']
)

#replacing blank with actual blanks
#award_with_recipient.replace("<blank>","")
award_with_recipient.replace("<blank>", "")

award_with_recipient.head()

In [ ]:
# Remove all schemes under departments listed in redaction file

In [ ]:
# Remove all schemes under departments listed in redaction file

depts_for_removal = pd.read_excel(r'..\data\\\redacted_fields FY23-24.xlsx', 2)

for index, row in depts_for_removal.iterrows():
    scheme_df = scheme_df[scheme_df['Funder: Department abbreviation'] != row['Department Abbrev']]

# Remove schemes as listed in redaction file
schemes_for_removal = pd.read_excel(r'..\data\\\redacted_fields FY23-24.xlsx', 0)

for index, row in schemes_for_removal.iterrows():
    scheme_df = scheme_df[scheme_df['Scheme Reference #'] != row['Scheme Reference']]

In [ ]:
award_with_recipient['Award approved date'] = award_with_recipient['Award approved date'].fillna(start_of_fy)  # missing award approval dates go to start of FY

award_with_recipient['Last modified'] = award_with_recipient['Last Modified Date'].apply(
    lambda x: datetime.strptime(x, "%d/%m/%Y").strftime("%Y-%m-%d") + "T00:00:00Z"
)

award_with_recipient['Award Date'] = award_with_recipient['Award approved date'].apply(
    lambda x: datetime.strptime(str(x), "%d/%m/%Y").strftime("%Y-%m-%d")
)

In [ ]:
award_with_recipient.loc[
    award_with_recipient['Do not publish recipient address'] == "1",
    ['Recipient Org:Street Address']
] = "Redacted"

award_with_recipient.loc[
    award_with_recipient['Do not publish recipient address'] == "1",
    ['Recipient Org:City']
] = "Redacted"

award_with_recipient.loc[
    award_with_recipient['Do not publish recipient address'] == "1",
    ['Recipient Org:Country']
] = "Redacted"

award_with_recipient.loc[
    award_with_recipient['Do not publish recipient address'] == "1",
    ['Recipient Org:Postal Code']
] = "Redacted"

boop = award_with_recipient

In [ ]:
award_with_recipient = award_with_recipient[[
    'Identifier',
    'Title',
    'Description',
    'Currency',
    'Amount Awarded',
    'Grant Programme:Code',
    'Grant Programme:Title',
    'Award Date',
    'Recipient Org:Identifier',
    'Recipient Org:Name',
    'Recipient Org:Charity Number',
    'Recipient Org:Company Number',
    'Recipient Org:Street Address',
    'Recipient Org:City',
    'Recipient Org:Country',
    'Recipient Org:Postal Code',
    'Funding Org:Identifier',
    'Funding Org:Name',
    'Managed by: Organisation Name',
    'Allocation Method',
    'From an open call?',
    'Authority Act',
    'Last modified',
    'Award type',
    'Number of recipients',
    'sort_on',
    'Recipient name at award creation'
]]

In [ ]:
print(award_with_recipient['Recipient Org:Country'].value_counts())

address_redacts = award_with_recipient[
    award_with_recipient['Recipient Org:Postal Code'] == "Redacted"
].reset_index(drop=True)

In [ ]:
award_with_recipient = award_with_recipient[[
    'Identifier',
    'Title',
    'Description',
    'Currency',
    'Amount Awarded',
    'Grant Programme:Code',
    'Grant Programme:Title',
    'Award Date',
    'Recipient Org:Identifier',
    'Recipient Org:Name',
    'Recipient Org:Charity Number',
    'Recipient Org:Company Number',
    'Recipient Org:Street Address',
    'Recipient Org:City',
    'Recipient Org:Country',
    'Recipient Org:Postal Code',
    'Funding Org:Identifier',
    'Funding Org:Name',
    'Managed by: Organisation Name',
    'Allocation Method',
    'From an open call?',
    'Authority Act',
    'Last modified',
    'Award type',
    'Number of recipients',
    'sort_on',
    'Recipient name at award creation'
]]

award_with_recipient.columns

In [ ]:
award = award_with_recipient.drop_duplicates(
    subset=["Identifier"]
).reset_index(drop=True)

print(len(award))

In [ ]:
award = award.sort_values(by=["sort_on"]).reset_index(drop=True)

In [ ]:
award = award[['Identifier','Title','Description','Currency','Amount Awarded','Grant Programme:Code','Grant Programme:Title',
               'Award Date','Recipient Org:Identifier','Recipient Org:Name','Recipient Org:Charity Number',
               'Recipient Org:Company Number','Recipient Org:Street Address','Recipient Org:City','Recipient Org:Country',
               'Recipient Org:Postal Code','Funding Org:Identifier','Funding Org:Name','Managed by: Organisation Name','Allocation Method',
               'From an open call?','Authority Act','Last modified',
               'Award type',
               'Number of recipients','Recipient name at award creation']]

In [ ]:
#convert fields to correct format, so they will be in correct format in excel:

#Date fields (as dates)
award['Award Date'] = pd.to_datetime(award['Award Date'], format='%Y-%m-%d').dt.date

#commented out this for 'Last modified date' - 360giving advised this needs to have the time stamp format, which is added earlier in this script
#award['Last modified'] = pd.to_datetime(award['Last modified'], format='%Y-%m-%d').dt.date

#Number fields (as numbers)
award['Amount Awarded'] = pd.to_numeric(award['Amount Awarded'])
award['Number of recipients'] = pd.to_numeric(award['Number of recipients'])

In [ ]:
def redact_id(row):
    if row['Recipient Org:Name'] == 'Redacted':
        return row['Identifier']
    elif row['Recipient Org:Company Number'] == 'Redacted':
        return row['Identifier']
    elif row['Recipient Org:Charity Number'] == 'Redacted':
        return row['Identifier']
    else:
        return(row['Recipient Org:Identifier'])

award['Recipient Org:Identifier'] = award.apply(redact_id, axis = 1)

# award['name_distance'] = award.apply(lambda x: Levenshtein.distance(str(x['Recipient Org:Name']).lower(), str(x['Recipient name at award creation']).lower()), axis=1)

In [ ]:
award.to_excel(writer, sheet_name='#Awards', index=False)

In [ ]:
# Get the xlsxwriter workbook and worksheet objects.
workbook = writer.book
worksheet = writer.sheets['#Awards']

# Add some cell formats.
num_format = workbook.add_format({'num_format': '0.00'})
date_format = workbook.add_format({'num_format': 'yyyy-mm-dd'})
int_format = workbook.add_format({'num_format': '0'})

# Note: It isn't possible to format any cells that already have a format such
# as the index or headers or any cells that already contain dates or datetimes.

# Set the format (4th column is the amount)
worksheet.set_column(4, 4, None, num_format)

# Set the format (8th column is the award date)
worksheet.set_column(8, 8, None, date_format)

# Set the format (23rd column is the Last modified date)
worksheet.set_column(23, 23, None, date_format)

# Set the format (24th column is the number of recipients)
worksheet.set_column(24, 24, None, int_format)

In [ ]:
Meta = pd.read_excel(r'..\data\Meta Data FY23-24.xlsx', 'Meta', index_col=None)
Meta = Meta.fillna('')
###worksheet = workbook.add_worksheet('Meta')
worksheet = workbook.get_worksheet_by_name('Meta')

worksheet.set_column('A:A', 25)

#Data for left column
row=1
wrap_format = workbook.add_format({'text_wrap': True})
red_format = workbook.add_format({'font_color': 'red'})
for i in range(len(Meta)):
    worksheet.write(row, 0, Meta.values[i][0])
    row+=1

url_format = workbook.get_default_url_format()

#Cell-by-Cell
# Add A1
worksheet.write('A1', "#")
worksheet.write('B1', "hashComments")
worksheet.write('B2', version)
worksheet.write('B3', 'Government_grants_register_2021_to_2022.ods')
worksheet.write('B4', f"This file contains data on UK Exchequer funded grant schemes and awards, active during the financial year {yyy}.")
worksheet.write('B5', '2023-03-30')
worksheet.write('B6', 'GGR.2023.1.0')
worksheet.write('B7', 'Cabinet Office')
worksheet.write('B8', '=HYPERLINK("https://www.gov.uk/government/organisations/cabinet-office", "https://www.gov.uk/government/organisations/cabinet-office")', url_format)
worksheet.write('B9', 'Image DBC', red_format)
worksheet.write('B10', 'GB-GOR-D2')
worksheet.write('B11', '=HYPERLINK("https://www.gov.uk/government/statistics/government-grants-statistics-2021-to-2022", "https://www.gov.uk/government/statistics/government-grants-statistics-2021-to-2022")', url_format)
worksheet.write('B12', '2023-03-30')
worksheet.write('B13', '=HYPERLINK("http://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/", "http://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/")', url_format)
worksheet.write('B14', 'grants.data@cabinetoffice.gov.uk')
worksheet.write('B15', '=HYPERLINK("https://www.gov.uk/government/statistics/government-grants-statistics-2021-to-2022", "https://www.gov.uk/government/statistics/government-grants-statistics-2021-to-2022")', url_format)
worksheet.write('B16', f"Further detail on data limitations and departmental statements about this data can be found in the Government grants statistics {yyy} at the URL above.")

In [ ]:
award['Award type']

general_awards = award[award['Award type'] == 'Organisation']
general_awards = general_awards[general_awards['Allocation Method'] != 'Formula'].reset_index(drop=True)

acronyms = department_acronym['Acronym'].unique()
acronyms = sorted(acronyms, key=str.casefold)

funders = general_awards['Funding Org:Name'].unique()

for i in acronyms:
    for j in range(len(funders)):
        if department_acronym.at[j,'Acronym'] == i:
            k = department_acronym.at[j,'Funder: Organisation Name']
            general_awards[general_awards['Funding Org:Name'] == k].to_excel(
                writer,
                sheet_name=f'{i}',
                index=False
            )
            workbook = writer.book
            worksheet = writer.sheets[f'{i}']
            num_format = workbook.add_format({'num_format': '0.00'})
            worksheet.set_column(4, 4, None, num_format)

In [ ]:
workbook = writer.book

worksheet = workbook.get_worksheet_by_name('#Contents')

worksheet.set_column('A:A', 30)
worksheet.set_column('B:B', 90)

worksheet.write_url('A10', f"internal:'#Schemes'!A1", string='#Schemes')
worksheet.write_url('A11', f"internal:'#Awards'!A1", string='#Awards')
worksheet.write_url('A12', f"internal:'#Scheme_field_notes'!A1", string='#Scheme_field_notes')
worksheet.write_url('A13', f"internal:'#Award_field_notes'!A1", string='#Award_field_notes')
worksheet.write_url('A14', f"internal:'Meta'!A1", string='Meta')
worksheet.write('A15', 'Department general awards tabs')

worksheet.write('B10', f"This tab contains the full scheme level data for {yyy} general and formula schemes.")
worksheet.write('B11', f"This tab contains the full award level data for {yyy} general and formula awards, including aggregated awards to individuals.")
worksheet.write('B12', f"This tab describes each field in the schemes tab and corresponding caveats.")
worksheet.write('B13', f"This tab describes each field in the awards tab and corresponding caveats.")
worksheet.write('B14', f"This tab provides meta-data about this file.")
worksheet.write('B15', f"The remaining tabs contain general awards data for each department, excluding awards going to individuals. \nThis is to ensure compliance with 360Giving Standard.",
    wrap_format
)

In [ ]:
writer.save()
writer.close()

In [ ]:
#copying sheets from scheme & award field notes file into register (Award notes tab):
#note - this cell will only work if you have excel installed on your PC
#error will occur if you have the file open when running this cell

import xlwings as xw

...

path1 = r'..\data\2022-12-21 scheme & award field notes (fy21-22 publication).xlsx'
path2 = r'..\data\\' + today + '_Government_Grants_Register_exports\\' + today + '_Government_Grants_Register_2023 to 2024.xlsx'

...

path1 = r'..\data\2025-03-18 scheme & award field notes (fy23-24 publication).xlsx'
print(path1)
path2 = r'..\data\\' + today + '_Government_Grants_Register_exports\\' + today + '_Government_Grants_Register_2023 to 2024.xlsx'
print(path2)

wb1 = xw.Book(path1)
wb2 = xw.Book(path2)

ws1 = wb1.sheets[2]
ws1.api.Copy(Before=wb2.sheets(2).api)
wb2.save()
wb2.app.quit()

In [ ]:
#Folder to save the figures
signoff = 'Signoff files'
if not os.path.exists(folder+'/' +signoff):
    os.makedirs(folder+'/' +signoff)

In [ ]:
path_formula = folder + '/' + signoff + '/' + today + '-scheme_register_df.csv'
path_general = folder + '/' + signoff + '/' + today + '-award_register_df.csv'

scheme.to_csv(path_formula, index=False, encoding=CSV_OUT_ENCODING)
award.to_csv(path_general, index=False, encoding=CSV_OUT_ENCODING)

In [ ]:
#Creating the top 10 dataframe
top_10 = pd.concat([general_df[:10], formula_df[:10]], ignore_index=True)

In [ ]:
#Replace department acronyms with correctly formatted acronyms with lower case letters where required, important for creating top 10 tables as it merges
top_10['Funder: Department abbreviation'] = top_10['Funder: Department abbreviation'].replace({
    'DFE': 'DfE',
    'DfT': 'DfT',
    'DEFRA': 'Defra',
    'MOJ': 'MoJ',
    'MOD': 'MoD'
})

In [ ]:
print(len(scheme_df))
scheme_df = scheme_df[scheme_df['Public funding source']=='UK Exchequer'].reset_index()
print(len(scheme_df))

In [ ]:
#check nans in scheme df
testnans = scheme_df[scheme_df['FY value'].isnull()]
testnans

In [ ]:
# Create new column for purpose and objective replacing text characters to ensure exports will be successful
scheme_df['Purpose and objectives'] = scheme_df['Scheme aims and objectives'] # rename column

for i in range(len(scheme_df)):
    if type(scheme_df.at[i,'Purpose and objectives']) == str:
        scheme_df.at[i,'Purpose and objectives'] = scheme_df.at[i,'Purpose and objectives'].replace("\x92", "'")
        scheme_df.at[i,'Purpose and objectives'] = scheme_df.at[i,'Purpose and objectives'].replace("\n", " ")
        scheme_df.at[i,'Purpose and objectives'] = scheme_df.at[i,'Purpose and objectives'].replace("Â", "")

In [ ]:
#make a dictionary from the department acronym dataframe to apply to the data
department_acronym = read_csv_clean(r'..\data\2021-01-01-DepartmentAcronyms.csv', dtype='object')
acronym = dict(zip(department_acronym['Funder: Organisation Name'], department_acronym['Acronym']))


In [ ]:
#scheme_df['Funder: Department abbreviation'] = scheme_df['Funder: Department abbreviation'].replace({'Department for Digital, culture, media, & sport': 'Department for Culture, Media and Sport'})

In [ ]:
for i in range(len(scheme_df)):
    if scheme_df.at[i,'Funder: Organisation Name'] == 'Department for Digital, Culture, Media & Sport':
        scheme_df.at[i,'Funder: Organisation Name'] = 'Department for Culture, Media & Sport'

In [ ]:
scheme_df['Funder: Department abbreviation'] = scheme_df['Funder: Department abbreviation'].replace({
    'Department for Digital, Culture, Media & Sport':
    'Department for Culture, Media & Sport'
})

In [ ]:
scheme_df['Department'] = scheme_df['Funder: Department abbreviation'].replace({
    'Department for Digital, culture, media, & sport':
    'Department for Culture, Media & Sport'
})

In [ ]:
#Create column of acronyms in data set utilising dictionary and dealing with exceptions
#df = df[df['Funder: Organisation Name']!='British Film Institute'].reset_index()
########## comment this out as this update is implemented on GGIS ############################################

for i in range(len(scheme_df)):
    if scheme_df.at[i,'Funder: Organisation Name'] == 'Ministry of Housing, Communities & Local Government':
        scheme_df.at[i,'Funder: Organisation Name'] = 'Department for Levelling Up, Housing and Communities'

for i in range(len(scheme_df)):
    if scheme_df.at[i,'Funder: Organisation Name'] == 'Department for Culture, Media and Sport':
        scheme_df.at[i,'Funder: Organisation Name'] = 'Department for Digital, Culture, Media & Sport'

def map_values(row, values_dict):
    return acronym[row]

scheme_df['Department'] = scheme_df['Funder: Organisation Name'].apply(map_values, args=(acronym,))

In [ ]:
# Remove Grant in Aid data from the dataframe
scheme_df = scheme_df[scheme_df['Allocation method']!='Grant in Aid']

In [ ]:
folder = f"../data/" + today + '_Statistics_Bulletin_script_exports'
if not os.path.exists(folder):
    os.makedirs(folder)

In [ ]:
# Create scheme and award dataframe
award_df = award_df.dropna(subset=['Awards FY breakdown Name'])

# Create scheme and award dataframe for the relevant year
scheme_df = scheme_df[scheme_df['Scheme FY breakdown Name'].str.contains(yyyy)].reset_index(drop=True)
award_df = award_df[award_df['Awards FY breakdown Name'].str.contains(yyyy)].reset_index(drop=True)

scheme_df = scheme_df.rename(columns={
    "FY value": "Scheme Value per year",
    "Scheme name": "Scheme Name",
    "Scheme Reference #": "System Scheme Reference No"
})

#for award data, need to create combined column within script:
award_df['Amount Awarded'] = np.where(
    (award_df['Amount paid'] == 0) |
    (award_df['Amount paid'].isnull()) |
    (award_df['Amount paid'] == '0'),
    np.where(
        (award_df['Amount paid'] == '0.00'),
        award_df['Budgeted Agreement value FY'],
        award_df['Amount paid']
    )
)

award_df = award_df.rename(columns={
    "Amount Awarded": "Total amount per year",
    "Award type": "Record Type.1"
})

#remove £ symbol and commas from values, so they can be converted to numeric dtype:
#scheme_df['Scheme Value per year'] = scheme_df['Scheme Value per year'].str.replace('£','')
#scheme_df['Scheme Value per year'] = scheme_df['Scheme Value per year'].str.replace(',','')
#award_df['Total amount per year'] = award_df['Total amount per year'].str.replace('£','')
#award_df['Total amount per year'] = award_df['Total amount per year'].str.replace(',','')

scheme_df['Scheme Value per year'] = pd.to_numeric(scheme_df['Scheme Value per year'])
award_df['Total amount per year'] = pd.to_numeric(award_df['Total amount per year'])

In [ ]:
ex_by_f = scheme_df[[
    "System Scheme Reference No",
    "Expenditure by function",
    "Scheme status"
]]

award_df = pd.merge(
    award_df,
    ex_by_f,
    how = "left",
    left_on = "Grant Scheme: Scheme Reference #",
    right_on = "System Scheme Reference No"
)

award_df = award_df[award_df['Scheme status'] != "Withdrawn"]
award_df = award_df[pd.notnull(award_df['Scheme status'])]

scheme_df[scheme_df['Expenditure by function'].isnull()]

award_df['cofog'] = award_df['Expenditure by function']
scheme_df['cofog'] = scheme_df['Expenditure by function']

In [ ]:
#insert facts that require information from outside data set or individual discretion
total_grant_value = int(round(sum(pd.to_numeric(scheme_df['Scheme Value per year'])) / 1000000000))
grant_percentage_of_expenditure = int(round(total_grant_value / total_government_expenditure * 100, 0))  #the percentage of government expenditure
#grants accounts for

In [ ]:
def total_value_bn(data, award_or_scheme, decimal=False):
    #function that takes a dataframe and sums the total value per year of all schemes in data set in billions
    #data - the data set to be analysed
    #award_or_scheme - whether the value to be returned is with respect to scheme value or award value

    if award_or_scheme == "award":
        if decimal == 1:
            return round(data['Total amount per year'].sum() / 1000000000, 1)
        else:
            return int(round(data['Total amount per year'].sum() / 1000000000, 0))
    else:
        if decimal == 1:
            return round(data['Scheme Value per year'].sum() / 1000000000, 1)
        else:
            return int(round(data['Scheme Value per year'].sum() / 1000000000, 0))

In [ ]:
def ordered_data(data, by):
    # function that orders data in descending order with respect to departments, schemes, cofogs or awards
    #data - the data set to be analysed
    #by- how the data is grouped and ordered
    if by == 'department':  #schemes are grouped by department then ordered in descending order by total scheme value of each department
        ordered = data.groupby(['Department', 'Funder: Organisation Name'])[['Scheme Value per year']].agg('sum') \
            .sort_values(by=['Scheme Value per year'], ascending=False).reset_index(drop=False)
        return ordered
    elif by == 'scheme':  #schemes ordered in descending order by scheme value
        ordered = data.sort_values(by=['Scheme Value per year'], ascending=False).reset_index(drop=True)
        return ordered
    elif by == 'cofog':  #awards are grouped by cofog then ordered in descending order by total award value of cofog
        ordered = data.groupby(['cofog'])[['Scheme Value per year']].agg('sum').sort_values(
            by=['Scheme Value per year'], ascending=False
        ).reset_index(drop=False)
        return ordered
    elif by == 'cofog_l1':  #awards are grouped by cofog_l1 then ordered in descending order by total award value of cofog_l1
        ordered = data.groupby(['cofog'])[['Scheme Value per year']].agg('sum').sort_values(
            by=['Scheme Value per year'], ascending=False
        ).reset_index(drop=False)
        return ordered
    else:  #awards ordered in descending order by award value
        ordered = data.sort_values(by=['Total amount per year'], ascending=False).reset_index(drop=False)
        return ordered


def specific_value(data, by, column, row, decimal=False):
    #function that takes a data set, utilising the ordered_data function then returns the value of a specific cell, if numerical in billions
    #data - the data set that is to be analysed
    #by- how the data is to be grouped and ordered
    #column- the name of the column of the cell to be returned
    #row- in decending order the position of the department, cofog, scheme or award starting from 1 (not 0)
    ordered = ordered_data(data, by)
    if type(ordered.loc[row-1, column]) != str:  #if data is numerical return value in billions
        if decimal == 1:
            return round(ordered.loc[row-1, column] / 1000000000, 1)
        else:
            return int(round(ordered.loc[row-1, column] / 1000000000, 0))
    else:
        return ordered.loc[row-1, column]


def combined_value(data, rows, department_or_scheme='department', decimal=False):
    #Total value of a given number of departments, which are ordered by total value of grants
    #data - the data set that is to be analysed
    #rows- in descending order the number of rows to be included in calculation
    #department_or_scheme - department if calculating combined value of department same for scheme
    if department_or_scheme == 'department':
        ordered = data.groupby(['Department', 'Funder: Organisation Name'])[['Scheme Value per year']].agg('sum') \
            .sort_values(by=['Scheme Value per year'], ascending=False).reset_index()
    else:
        ordered = ordered_data(data, by='scheme')

    if decimal == 1:
        return '£' + str(round(ordered.head(rows)['Scheme Value per year'].sum() / 1000000000, 1)) + ' billion'
    else:
        return '£' + str(int(round(ordered.head(rows)['Scheme Value per year'].sum() / 1000000000, 0))) + ' billion'


def percentage_value(data, departments):
    #Percentage of the total value that a given number of departments contribute
    #data - the data set that is to be analysed
    #departments - the number of departments to include in the numerator in descending order
    top_departments = data.groupby(['Department', 'Funder: Organisation Name'])[['Scheme Value per year']].agg('sum') \
        .sort_values(by=['Scheme Value per year'], ascending=False).reset_index()

    return int(
        round(
            top_departments.head(departments)['Scheme Value per year'].sum()
            / data['Scheme Value per year'].sum(),
            2
        ) * 100
    )

In [ ]:
def percentage_of_total(data):
    return str(
        int(
            round(
                sum(data['Scheme Value per year']) /
                sum(scheme_df['Scheme Value per year']) * 100,
                0
            )
        )
    )

In [ ]:
def departments_by_value(data):
    #function that orders departments by value
    #data - the data set that is to be analysed
    data = data.groupby(
        ['Department', 'Funder: Organisation Name']
    )[['Scheme Value per year']].agg('sum') \
        .sort_values(
            by=['Scheme Value per year'],
            ascending=False
        ).reset_index()

    data['Scheme Value per year'] = data['Scheme Value per year'].astype(float)

    return data

In [ ]:
def cofog_by_value(data):
    #function that orders cofog by value
    #data - the data set that is to be analysed
    data = data.groupby(
        ['cofog']
    )[['Scheme Value per year']].agg('sum').sort_values(
        by=['Scheme Value per year'],
        ascending=False
    ).reset_index()

    data['Scheme Value per year'] = data['Scheme Value per year'].astype(float)

    return data

In [ ]:
#def allocation_method(data, rows)

In [ ]:
def median_value(data, award=False):
    #function that calculates the median value of a set of data
    #data - the data set that is to be analysed
    if award == True:
        return '£' + str(
            int(round(data['Total amount per year'].median() / 1000, 0))
        ) + ' thousand'
    else:
        if data['Scheme Value per year'].median() < 1000000000:
            return '£' + str(
                round(data['Scheme Value per year'].median() / 1000000, 0)
            ) + ' million'
        else:
            return '£' + str(
                int(round(data['Scheme Value per year'].median() / 1000000000, 0))
            ) + ' billion'

In [ ]:
def median_value_numerical(data, award=False):
    #function that calculates the median value of a set of data
    #data - the data set that is to be analysed
    if award == True:
        return round(data['Total amount per year'].median() / 1000, 0)
    else:
        if data['Scheme Value per year'].median() < 1000000000:
            return round(data['Scheme Value per year'].median() / 1000000, 0)
        else:
            return round(data['Scheme Value per year'].median() / 1000000000, 0)

In [ ]:
def percentage_of_total_grants_by_value(data, scheme_or_award):
    #function that calculates the percentage of grants are account for by a data set (by value)
    #data - the data set that is to be analysed
    #scheme_or_award - is the data scheme or award data (the denominator)

    if scheme_or_award == 'scheme':
        result = data['Scheme Value per year'].sum() / scheme_df['Scheme Value per year'].sum()
        print(result)
        return int(np.round(np.round(result, 2) * 100, 0))

    elif scheme_or_award == 'general':
        return int(
            round(
                data['Total amount per year'].sum() /
                general_a_df['Total amount per year'].sum(),
                2
            ) * 100
        )

    else:
        return int(
            round(
                data['Total amount per year'].sum() /
                award_df['Total amount per year'].sum(),
                2
            ) * 100
        )

In [ ]:
def approximate_proportion_of_total_grants_by_value(data1, data2, rows):
    #function that outputs an approximate proportion of the total grants value that is accounted for by (rows) number of the top schemes
    #data1 - the data set that is to be analysed (numerator)
    #data2 - the data set that is to be analysed (denominator), eg whether the full scheme data set or just formula data
    #rows - the number of schemes to be analysed in the numerator

    ordered1 = ordered_data(data=data1, by='scheme')
    ordered2 = ordered_data(data=data2, by='scheme')

    subset = ordered1.head(rows)['Scheme Value per year'].sum() / 1000000000
    fullset = ordered2['Scheme Value per year'].sum() / 1000000000
    percentage = subset / fullset * 100

    if percentage > 50:
        return 'over half'
    elif percentage > 100/3:
        return 'over a third'
    elif percentage > 25:
        return 'over a quarter'
    elif percentage > 20:
        return 'over a fifth'
    else:
        return 'around ' + str(int(percentage)) + '%'

In [ ]:
def percentage_of_total_grants_by_quantity(data):
    #data - the data set that is to be analysed
    return int(round(len(data) / len(scheme_df) * 100, 0))

In [ ]:
def set_size(w, h, ax=None):
    #set size of figures - w = width, h = height
    """ w, h: width, height in inches """
    if not ax:
        ax = plt.gca()

    l = ax.figure.subplotpars.left
    r = ax.figure.subplotpars.right
    t = ax.figure.subplotpars.top
    b = ax.figure.subplotpars.bottom

    figw = float(w) / (r-l)
    figh = float(h) / (t-b)

    ax.figure.set_size_inches(figw, figh)

In [ ]:
def scheme_tables(data, rows):
    #data - the data set that is to be analysed
    data = data.sort_values(
        by=['Scheme Value per year'],
        ascending=False
    ).head(rows).reset_index(drop=True)

    data['Value'] = '£' + round(
        data['Scheme Value per year'] / 1000000000,
        1
    ).apply(str) + ' billion'

    table1 = data[[
        'Department',
        'Scheme Name',
        'Purpose and objectives',
        'Value'
    ]]

    return table1

In [ ]:
def chart(
    department_or_scheme,
    data1,
    data2,
    title,
    excel_sheet_name=False,
    merged_field=False,
    field2=False,
    rows=False,
    stacked=False,
    allocation_method=False,
    svg=False,
    false,
    file_name=False,
    formula_or_general=False,
    table_name=False
):
    # department_or_scheme - is the output relate to scheme, department, cofog or allocation method
    # data1 - if only one data set utilises this, if two data sets eg general and formula combining into stacked barchart its the data listed first
    # data2 - if only one data set leave blank, if two data sets eg general and formula combining into stacked barchart its the data listed second
    # title - the title of the output
    # excel_sheet_name - the name of the sheet that data will be saved to
    # merged_field - if merging two data sets the field the field that will remain in the table, eg the departments
    # field2 - if merging two data sets the field with the two comparable statistics, eg scheme value
    # rows - the number of rows to include in the output
    # stacked - is it a stacked barchart - NOT APPLICABLE to 'allocation method'
    # allocation_method - the analysis undertaken on the allocation method eg median or sum


#Creation of the relevant dataframes
	if department_or_scheme == 'department':  #department data with both formula and general
		if stacked == True:
			data = pd.merge(
				departments_by_value(data1)[[merged_field, field2]],
				departments_by_value(data2)[[merged_field, field2]],
				on=merged_field,
				how='outer'
			).fillna(0)

			data['Total'] = data[field2 + '_x'] + data[field2 + '_y']
			data = data.rename(columns={
				field2 + '_x': 'Formula',
				field2 + '_y': 'General'
			})

			data['Formula'] = data['Formula'].astype(float)
			data['General'] = data['General'].astype(float)
			data['Total'] = data['Total'].astype(float)

			data = data.sort_values(
				by=['Total'],
				ascending=False
			).reset_index(drop=True)

		else:  # department data with either formula or general
			data = departments_by_value(data1)[[merged_field, field2]]
		
	elif department_or_scheme == 'scheme':  # scheme data
		print(data1)
		print(top_10)

		data1['Value'] = data1['Scheme Value per year']
		data1 = data1[data1['Do not publish'] != 1.0]

		data = data1[[
			'Department',
			'System Scheme Reference No',
			'Scheme Name',
			'Purpose and objectives',
			'Value'
		]].sort_values(
			by=['Value'],
			ascending=False
		).head(rows).reset_index(drop=True)

		data = data[[
			'Department',
			'System Scheme Reference No',
			'Scheme Name',
			'Purpose and objectives',
			'Value'
		]].reset_index(drop=True)

		data = data.merge(
			top_10,
			how='left',
			left_on=['Department', 'System Scheme Reference No'],
			right_on=['Funder: Department abbreviation', 'Scheme Reference #']
		)

		#Note - department acronyms must have the same case to merge properly, e.g. DFE and DfE will not merge correctly
		print(data)

		data = data[[
			'Department',
			'Scheme Name',
			'Purpose and objectives',
			'Value'
		]]

		#Format Data -- THIS DOESNT WORK!
		#data['Scheme Name'] = data['Scheme Name'].replace('\n',' ').replace('\r',' ')
		#data['Purpose and objectives'] = data['Purpose and objectives'].replace('\n',' ').replace('\r',' ')

		#data = data[['Department','Rewritten Scheme Name','Rewritten Purpose and objectives','Value_y']]
		#data = data.rename(columns={
		#    'Rewritten Scheme Name':'Scheme Name',
		#    'Rewritten Purpose and objectives':'Purpose and objectives',
		#    'Value_y':'Value'
		#})

		if len(data) < 10:
			print('ERROR')		
		
	elif department_or_scheme == 'cofog':  #cofog data
		data1 = cofog_by_value(formula_s_df)[[merged_field, field2]]
		data2 = cofog_by_value(general_s_df)[[merged_field, field2]]

		data = pd.merge(
			data1,
			data2,
			on=merged_field,
			how='outer'
		).fillna(0)

		data['Total'] = data[field2 + '_x'] + data[field2 + '_y']

		data = data.rename(columns={
			field2 + '_x': 'Formula',
			field2 + '_y': 'General'
		})

		data = data.sort_values(
			by=['Total'],
			ascending=False
		).reset_index(drop=True)

		data = data[['cofog', 'General', 'Formula', 'Total']]

		#data = data.rename(columns={"Total amount per year": "Total grant spending"})		
		
		
		
			
			
	elif department_or_scheme == 'allocation_method':  #allocation method data
		data1.reset_index(drop=True)

		for i in range(len(data1)):  # Removing all text before and including ' - '
			if ' - ' in data1.at[i, 'Allocation method']:
				data1.at[i, 'Allocation method'] = data1.at[i, 'Allocation method'].split(' - ', 1)[1]

		if allocation_method == 'sum':  #calculating the total for allocation method, and percentage
			data = data1.groupby(
				['Allocation method']
			)[['Total amount per year']].agg('sum').sort_values(
				by=['Total amount per year'],
				ascending=False
			).reset_index()

			data = data.rename(columns={
				"Total amount per year": "Total award value"
			})

			data = data.rename(columns={"Total amount per year": "Total award value"})
			data['Percentage of total general grant award value'] = 0

			for i in range(len(data)):
				data.at[i, 'Percentage of total general grant award value'] = int(
					round(
						data.at[i, 'Total award value'] * 100 /
						sum(data["Total award value"]),
						0
					)
				)

		elif allocation_method == 'median':  # calculating the median
			data = data1.groupby(
				['Allocation method']
			)[['Total amount per year']].agg('median').sort_values(
				by=['Total amount per year'],
				ascending=False
			).reset_index()

			data = data.rename(columns={
				"Total amount per year": "Median award value"
			})

			print(data)
# exporting to excel, this is found in cell below
	excel_tables(data, excel_sheet_name, table_name, stacked)		

#manually creating svg graphs to insert into govspeak
	if svg == True:
		width = 0.75

		#department graphs
		if department_or_scheme == 'department':

			# stacked barchart
			if stacked == True:
				fig, ax = plt.subplots(frameon=False)

				data = data.sort_values(
					by=['Total'],
					ascending=True
				).reset_index(drop=True)

				labels = data['Department']
				column1 = data['Formula']
				column2 = data['General']

				ax.barh(
					labels,
					column1,
					width,
					label='Formula',
					color='#f47738'
				)

				ax.barh(
					labels,
					column2,
					width,
					left=column1,
					color='#1d70b8',
					label='General'
				)

				ax.set_frame_on(False)
				ax.get_xaxis().set_visible(False)

				ax.legend(
					loc='lower left',
					bbox_to_anchor=(.6, .04),
					ncol=1,
					fontsize=15
				)

				set_size(5, len(data)/2.5)

				size_font = 15
				ax.set_ylabel('Department', fontsize=size_font)

			# general or formula barcharts
			else:
				fig, ax = plt.subplots(frameon=False)

				data = data.sort_values(
					by=['Scheme Value per year'],
					ascending=True
				).reset_index(drop=True)

				data['Total'] = data['Scheme Value per year']
				labels = data['Department']
				column1 = data['Scheme Value per year']

				if formula_or_general == 'formula':
					ax.barh(
						labels,
						column1,
						width,
						label='Scheme Value per year',
						color='#f47738'
					)
					size_font = 14

				elif formula_or_general == 'general':
					ax.barh(
						labels,
						column1,
						width,
						label='Scheme Value per year',
						color='#1d70b8'
					)
					size_font = 14.5

				ax.set_frame_on(False)
				ax.get_xaxis().set_visible(False)

				set_size(5, len(data)/2.5)

				ax.set_ylabel('Department', fontsize=size_font)

			
#cofog barchart
		elif department_or_scheme == 'cofog':
			print(data)
			fig, ax = plt.subplots(frameon=False)
			data = data.sort_values(by=['Total'], ascending=True).reset_index(drop=True)
			labels = data['cofog']
			column1 = data['Formula']
			column2 = data['General']

			ax.barh(labels, column1, width, label='Formula', color='#f47738')
			ax.barh(labels, column2, width, left=column1, color='#1d70b8', label='General')
			ax.set_frame_on(False)
			ax.get_xaxis().set_visible(False)
			#ax.legend(loc='lower')
			ax.legend(loc='lower left', bbox_to_anchor=(.6, .04), ncol=1, fontsize=15)

			set_size(5, len(data)/2.5)
			size_font = 15
			ax.set_ylabel('Economic classification', fontsize=size_font)

#turn numerical data into billion labels or million labels depending on size
		for i, v in enumerate(data['Total']):
			if v > 999999999:
				plt.text(
					v + data.at[len(data)-1, 'Total']/35,
					i-.2,
					'£' + str(round(v/1000000000,1)) + " billion",
					color='black',
					fontsize=size_font
				)
			elif v > 999999:
				plt.text(
					v + data.at[len(data)-1, 'Total']/35,
					i-.2,
					'£' + str(int(round(v/1000000,0))) + " million",
					color='black',
					fontsize=size_font
				)
			elif v > 999:
				plt.text(
					v + data.at[len(data)-1, 'Total']/35,
					i-.2,
					'£' + str(int(round(v/1000,0))) + " thousand",
					color='black',
					fontsize=size_font
				)
			else:
				plt.text(
					v + data.at[len(data)-1, 'Total']/35,
					i-.2,
					'£' + str(int(round(v/10,0))),
					color='black',
					fontsize=size_font
				)

		ax.set_title(title, fontsize=size_font)
		plt.yticks(fontsize=size_font)
		plt.savefig(folder + '/' + blended + '/' + today + file_name, bbox_inches='tight')
		plt.show()			
		
#manually creating html for govspeak to be exported into a .txt
	elif svg == False:

		# creation of title line
		string = "####" + title + "\n\n"

		#creation of second line to create tables
		for i in data.columns:
			if i == data.columns[-1]:
				if len(data.columns) == 2:
					string += i + '\n -|-'
				elif len(data.columns) == 3:
					string += i + '\n -|-|-'
				elif len(data.columns) == 4:
					string += i + '\n -|-|-|-'
			else:
				string += i + '|'

		print(data.columns.isna().sum())

		#creation of contents of the table
		for i in range(len(data)):
			print(i)
			for j in range(len(data.columns)):
				print(j)
                if j == 0:
                    string += '\n' + data.at[i, data.columns[j]]
                elif type(data.at[i, data.columns[j]]) == str:
                    string += '|' + data.at[i, data.columns[j]]
                elif department_or_scheme == 'allocation method' and data.columns[j] == 'Median value per allocation method':
                    #exception for medina data formatting
                    string += '|' + '£' + f'{int(round(data.at[i,data.columns[j]]/1000,0))}' + 'k'
                elif department_or_scheme == 'allocation method' and data.columns[j] == 'Percentage of total general grant award value':
                    #exception for percentage data formatting
                    string += '|' + data.at[i,data.columns[j]].astype(str) + '%'
                else:
                    if data.at[i,data.columns[j]] > 999999999:
                        string += '|' + '£' + round(data.at[i,data.columns[j]]/1000000000,1).astype(str) + ' billion'
                    elif data.at[i,data.columns[j]] == 0:
                        0
                    elif pd.isna(data.at[i,data.columns[j]]):
                        0
                    else:
                        string += '|' + '£' + str(int(round(data.at[i,data.columns[j]]/1000000,0))) + ' million'

    print(string)

    #creation of the type of table
    if stacked == True:
        string += '\n {barchart stacked} \n\n'
    elif department_or_scheme == 'department':
        string += '\n {barchart} \n\n'
    elif department_or_scheme == 'scheme':
        string += '\n\n'
    elif allocation_method == 'sum':
        string += '\n {barchart stacked} \n\n'
    else:
        string += '\n {barchart} \n\n'

    return string				

In [ ]:
path0 = f"../data/" + folder + '/' + 'Government grants statistical tables ' + yyyy + '.xlsx'  #name of excel that will be exported
writer = pd.ExcelWriter(path0, engine = 'xlsxwriter')  #openning function that writes data to the open excel

In [ ]:
#Draft name of excel file
mydate = datetime.now()
mydate.strftime("%B") + ' ' + mydate.strftime("%Y")

data = pd.DataFrame(data = {
    f"Government grants statistical tables {yyy}": [
        f"Statistical tables - {mydate.strftime('%B')} " + ' ' + mydate.strftime('%Y')
    ],
    "Introduction": [
        f"This document contains the tables underlying the figures shown in the Government grants statistics {yyy} bulletin."
    ],
    f"The data from which these statistics are derived is also published alongside these tables, as the Government grants register {yyy}* - scheme and award data.": [
        "Contents"
    ],
    "Tab": [""]
})

In [ ]:
#Creating first tab - 'Contents'
workbook = writer.book
worksheet = workbook.add_worksheet('Contents')

In [ ]:
#Cell-by-Cell
# Add A1
A1 = workbook.add_format({'bold': True, 'font_size':24})
worksheet.write('A1', data.columns[0], A1)

#Add A2
A2 = workbook.add_format({'italic': True, 'font_size':11})
worksheet.write('A2', data.values[0][0], A2)

#Add A4
A4 = workbook.add_format({'bold': True, 'font_size':14})
worksheet.write('A4', data.values[1][0], A4)

#Add A5 and A6
worksheet.write('A5', data.values[2][0])
worksheet.write('A6', data.values[3][0])

#Add G5 and G6
worksheet.write('G5', data.values[2][0])
worksheet.write('G6', data.values[3][0])

#Add A9
worksheet.write('A9', data.values[4][0], A4)

#Add A10
A10 = workbook.add_format({'bold': True, 'italic': True, 'font_size':11})
worksheet.write('A10', data.values[5][0], A10)

#Add B10
worksheet.write('B10', 'Description', A10)

#hyperlinks to other sheets done after those sheets have been created

In [ ]:
# function for the excel sheets that contain data from charts on the gov.uk website
def excel_tables(data, excel_sheet_name=False, table_name=False, stacked=False):
    if excel_sheet_name != False:
        excel_sheet_name = excel_sheet_name.replace(" ", "_")
        workbook = writer.book

        if data.columns[[1]][0] == "Total award value": # exception for total award value data for the allocation method, this data needs to reopen sheet
            worksheet = workbook.get_worksheet_by_name(excel_sheet_name)
        else:
            worksheet = workbook.add_worksheet(excel_sheet_name)

        #formatting cells
        title_format = workbook.add_format({'bold':True, 'font_size': 18})
        sub_title_format = workbook.add_format({'italic':True, 'font_size': 11})
        currency_format = workbook.add_format({'italic':True, 'font_size': 11, 'align':'right'})
        header_format = workbook.add_format({'bold':True, 'font_size': 11, 'bottom': True, 'top': True, 'align':'right'})
        header_format2 = workbook.add_format({'bold':True, 'font_size': 11, 'bottom': True, 'top': True})
        numerical_format = workbook.add_format({'num_format': '#,##0_-;[Red]-#,##0_-;_-'})
        numerical_total_format1 = workbook.add_format({'bold':True, 'num_format': '#,##0_-;[Red]-#,##0_-;_-'})
        numerical_total_format = workbook.add_format({'bold':True, 'font_size':11, 'bottom':True, 'top':True, 'num_format': '#,##0_-;[Red]-#,##0_-;_-'})
        total_format = workbook.add_format({'bold':True, 'font_size':11, 'bottom':True, 'top':True})
        total_total_format = workbook.add_format({'bold':True, 'italic':True, 'font_size':11, 'bottom':True, 'top':True})
        total_total_numerical = workbook.add_format({'bold':True, 'italic':True, 'font_size':11, 'bottom':True, 'top':True, 'num_format': '#,##0_-;[Red]-#,##0_-;_-'})
        general_award_format = workbook.add_format({'bold':True, 'font_size':11, 'top':True})
        general_award_numerical = workbook.add_format({'bold':True, 'font_size':11, 'top':True, 'bottom':False, 'num_format': '#,##0_-;[Red]-#,##0_-;_-'})

        notes_format = workbook.add_format({'bold':True, 'font_size':11})

        #Cell-by-cell
        if data.columns[[1]][0] == "Total award value": # exception for total award value data for the allocation method, this data needs to reopen sheet
            worksheet.write('A1', '2 Value of grant awards by allocation method', title_format)
        else:
            worksheet.write('A1', table_name, title_format)
        worksheet.write('A2', 'All grants', sub_title_format)
		row = 3

		if excel_sheet_name == 'Table_2':
			worksheet.write('B4', '£ million', currency_format)
		else:
			worksheet.write(row, data.shape[1]-1, '£ million', currency_format)

		row += 1

		#Headings of the tables
		column = 0
		for i in data.columns:
			if i == 'Percentage of total general grant award value':  #exception, not to be included
				pass
			elif i == 'Total award value':  #exception needs to be one column to the right
				worksheet.write(row, column, i, header_format)
				worksheet.write(row, column-1, 'Allocation method', header_format2)
			elif i == 'Median award value':  #exception needs to be one column to the right
				pass
			elif column == 0:  # first column has text aligned on the left
				worksheet.write(row, column, i, header_format2)
			else:
				worksheet.write(row, column, i, header_format)
			column += 1

		#Data of the tables
		row += 1
		first_row = row
		print(data)

		if excel_sheet_name == 'Table_2':
			if data.columns[[1]][0] != 'Total award value':
				pass
			else:
				worksheet.write('A6', 'General', general_award_format)
				worksheet.write(
					'B6',
					int(round(general_a_df['Total amount per year'].sum()/1000000, 0)),
					general_award_numerical
				)
				worksheet.write('A7', '   ' + data.values[0][0])
				worksheet.write(
					'B7',
					int(round(data.values[0][1]/1000000, 0)),
					numerical_format
				)
				worksheet.write('A8', '   ' + data.values[1][0])
				worksheet.write(
					'B8',
					int(round(data.values[1][1]/1000000, 0)),
					numerical_format
				)
				worksheet.write('A9', '   ' + data.values[2][0])
				worksheet.write(
					'B9',
					int(round(data.values[2][1]/1000000, 0)),
					numerical_format
				)
				worksheet.write('A10', 'Formula', total_format)
				worksheet.write(
					'B10',
					int(round(formula_a_df['Total amount per year'].sum()/1000000, 0)),
					numerical_total_format
				)
				worksheet.write('A11', 'Total', total_total_format)
				worksheet.write(
					'B11',
					int(round(award_df['Total amount per year'].sum()/1000000, 0)),
					total_total_numerical
				)
				worksheet.write('A14', 'Notes:', notes_format)
				worksheet.write('A15', 'Values rounded to the nearest million.')
				worksheet.write(
					'A16',
					'These totals are calculated based on award level data. Therefore the figures will not match those shown elsewhere in this report.'
				)

		else:
			print(data)
			for i in range(len(data)):
				for j in range(data.shape[1]):
					if type(data.values[i][j]) != str and data.values[i][j] != 'Total award value' and data.values[i][j] != 'Percentage of total general grant award value':
						if j > 2:
							worksheet.write(
								row,
								j,
								int(round(data.values[i][j]/1000000, 0)),
								numerical_total_format1
							)
						else:
							worksheet.write(
								row,
								j,
								int(round(data.values[i][j]/1000000, 0)),
								numerical_format
							)
					#non numerical data
					else:
						worksheet.write(row, j, data.values[i][j])
				row += 1

			last_row = row - 1

			# Column widths to fit data, set arbitrarily
			worksheet.set_column('A:A', 20)
			worksheet.set_column('B:F', 18)

			# Total columns at bottom of tables
			column = 0
			if table_name == '2 Average and total value of general grant awards by general allocation method':
				pass

			else:
				for j in range(data.shape[1]):
					if j == 0:
						if excel_sheet_name == 'Table_2':
							pass
						else:
							worksheet.write(row, j, 'Total', total_format)
					else:
						worksheet.write(
							row,
							j,
							int(round(data.iloc[:, [j]].sum() / 1000000, 0)),
							numerical_total_format
						)

			# column width for cofog column changed specifically to fit data
			for i in data.columns:
				if i == 'cofog':
					worksheet.set_column('A:A', 30)

			if excel_sheet_name == 'Table_2':
				worksheet.write(
					'B6',
					int(round(general_a_df['Total amount per year'].sum() / 1000000, 0)),
					numerical_total_format
				)
				worksheet.write('B4', '')

			if excel_sheet_name == 'Table_1':
				worksheet.write('A24', 'Notes:', notes_format)
				worksheet.write('A25', 'Values rounded to the nearest million.')
				worksheet.write('A26', 'Department values might not sum to Total because of rounding.')

			if excel_sheet_name == 'Table_3':
				worksheet.write('A19', 'Notes:', notes_format)
				worksheet.write('A20', 'Values rounded to the nearest million.')
				worksheet.write('A21', 'Department and COFOG values might not sum to Total because of rounding.')

In [ ]:
issues = scheme_df[pd.isnull(scheme_df['Allocation method'])]
issues.head()

In [ ]:
#Scheme data
#formula scheme data
formula_df = scheme_df[scheme_df['Allocation method'] == 'Formula Grant'].reset_index(drop=True)

#general grants scheme data
general_df = scheme_df[pd.notnull(scheme_df['Allocation method'])]
general_df = general_df[general_df['Allocation method'].str.contains('General')].reset_index(drop=True)

#formula and scheme data of top general department
for_1_df = formula_df[
    formula_df['Funder: Organisation Name'] ==
    specific_value(formula_df, 'department', 'Funder: Organisation Name', 1)
]

for_one_df = scheme_df[
    scheme_df['Funder: Organisation Name'] ==
    specific_value(formula_df, 'department', 'Funder: Organisation Name', 1)
]

#formula and scheme data without top formula department
for_0_df = formula_df[
    formula_df['Funder: Organisation Name'] !=
    specific_value(formula_df, 'department', 'Funder: Organisation Name', 1)
]

for_zero_df = scheme_df[
    scheme_df['Funder: Organisation Name'] !=
    specific_value(formula_df, 'department', 'Funder: Organisation Name', 1)
]

#general and scheme data of top general department
gen_1_df = general_df[
    general_df['Funder: Organisation Name'] ==
    specific_value(general_df, 'department', 'Funder: Organisation Name', 1)
]

#general and scheme data without top general department
gen_0_df = general_df[
    general_df['Funder: Organisation Name'] !=
    specific_value(general_df, 'department', 'Funder: Organisation Name', 1)
]

gen_zero_df = scheme_df[
    scheme_df['Funder: Organisation Name'] !=
    specific_value(general_df, 'department', 'Funder: Organisation Name', 1)
]

#Award Data
#formula grants award data
formula_a_df = award_df[
    award_df['Allocation method'] == 'Formula'
].reset_index(drop=True)

#general grants award data
general_a_df = award_df[
    award_df['Allocation method'].str.contains('General')
].reset_index(drop=True)

general_a_df = general_a_df[
    general_a_df['Allocation method'].str.contains('General')
].reset_index(drop=True)

#general grants - competed award data
competed_df = award_df[
    award_df['Allocation method'] == 'General grants - Competed'
].reset_index(drop=True)

#competed_df_orgs = competed_df[competed_df['Record Type.1'] == "Organisation"].reset_index(drop=True)

#general grants - uncompeted award data
uncompeted_df = award_df[
    award_df['Allocation method'] == 'General grants - Un-competed'
].reset_index(drop=True)

#uncompeted_df_orgs = uncompeted_df[uncompeted_df['Record Type.1'] == "Organisation"].reset_index(drop=True)

#general grants - criteria award data
criteria_df = award_df[
    award_df['Allocation method'] == 'General grants - Criteria based'
].reset_index(drop=True)

criteria_df_orgs = criteria_df[
    criteria_df['Record Type.1'] == "Organisation"
].reset_index(drop=True)

#general grants - top cofog
cofog_1 = scheme_df[
    scheme_df['cofog'] == specific_value(scheme_df, 'cofog', 'cofog', 1)
]

#general grants - 2nd top cofog
cofog_2 = scheme_df[
    scheme_df['cofog'] == specific_value(scheme_df, 'cofog', 'cofog', 2)
]

scheme_df['Allocation method'] = scheme_df['Allocation method'].fillna('')

#formula grants scheme data
formula_s_df = scheme_df[
    scheme_df['Allocation method'] == 'Formula Grant'
].reset_index(drop=True)

#general grants awschemeard data
general_s_df = scheme_df[
    scheme_df['Allocation method'].str.contains('General')
].reset_index(drop=True)

In [ ]:
organisation_awards = award_df[
    award_df['Record Type.1'] == 'Organisation'
].reset_index()

organisation_formula_awards = formula_a_df[
    formula_a_df['Record Type.1'] == 'Organisation'
].reset_index()

organisation_general_awards = general_a_df[
    general_a_df['Record Type.1'] == 'Organisation'
].reset_index()

In [ ]:
criterias_df = general_df[
    general_df['Allocation method'] == 'General Grant - Criteria Based'
].reset_index(drop=True)

competeds_df = general_df[
    general_df['Allocation method'] == 'General Grant - Competed'
].reset_index(drop=True)

uncompeteds_df = general_df[
    general_df['Allocation method'] == 'General Grant - Uncompeted'
].reset_index(drop=True)

mixeds_df = general_df[
    general_df['Allocation method'] == 'General Grant - Mixed'
].reset_index(drop=True)

print(len(mixeds_df))
print(len(criterias_df))
print(len(competeds_df))
print(len(uncompeteds_df))

In [ ]:
#general grants scheme data
print(sum(general_df['Scheme Value per year']))

print(
    sum(criterias_df['Scheme Value per year']) +
    sum(competeds_df['Scheme Value per year']) +
    sum(uncompeteds_df['Scheme Value per year']) +
    sum(mixeds_df['Scheme Value per year'])
)

In [ ]:
# Analysis text that may change
#grants2 = f"This includes the {specific_value(for_one_df, by='scheme', column='Scheme Name', row=1)}, ...

In [ ]:
# Analysis text that may change
#grants2 = f"This includes the {specific_value(for_one_df, by='scheme', column='Scheme Name', row=1)}, the {specific_value(for_one_df, by='scheme',
# column = 'Scheme Name', row=2)} and the {specific_value(for_one_df, by='scheme', column = 'Scheme Name', row=3)},
# these three schemes account for a combined total of {combined_value(data=for_one_df, rows=3,department_or_scheme='scheme')}
#grants8 = f"Large grant giving departments also include {specific_value(scheme_df, 'department','Department',2)}, the {specific_value(scheme_df,
# 'department', 'Department',3)}, {specific_value(scheme_df, 'department','Department',4)} and {specific_value(scheme_df,
# 'department','Department',5)}, which together with {specific_value(scheme_df,'department','Department',1)} give out
# {combined_value(scheme_df,5,decimal=1)} ({percentage_value(scheme_df,5)}%) of the total value of government grants."

grants2 = f"This includes the Dedicated Schools Grant (DSG), the General Annual Grant (GAG), and the 16-19 Education Grant; these three schemes account for a combined total of £56.8 billion."

grants8 = f"Large grant giving departments also include the Department for Levelling Up Housing & Communities (DLUHC), HMRC, the Department for Business, Energy & Industrial Strategy (BEIS), and the Home Office (HO) which together with {specific_value(scheme_df, 'department', 'Department',1)} gave out {combined_value(scheme_df,5,decimal=1)} ({percentage_value(scheme_df,5)}%) of the total value of government grants."

formula2 = f"This includes the {specific_value(for_1_df, by='scheme', column = 'Scheme Name', row=1)}, the {specific_value(for_1_df, by='scheme',
    column = 'Scheme Name', row=2)} and the {specific_value(for_1_df, by='scheme', column = 'Scheme Name', row=3)}, these three schemes account
    for a combined total of {combined_value(data=for_1_df,rows=3,department_or_scheme='scheme')} ({approximate_proportion_of_total_grants_by_
    value(data1=for_1_df,data2=formula_df,rows=3)} of all formula grants funding by value)."

formula5 = f"Outside of the {specific_value(for_1_df, 'department','Department',1)}, the next largest formula grant schemes are the DLUHC's £5.8 billion Expanded Retail Discount and the HO's £4.8 billion Police Main Grant."

#should be grants2
general2 = f"This includes the CJRS and the SEISS; these two schemes account for a combined total of {combined_value(data=gen_1_df,rows = 2,department_or_scheme='scheme', decimal=1)}."

general5 = f"Outside of {specific_value(gen_1_df, 'department', 'Department',1)}, the next largest general grant schemes are the {specific_value(gen_0_df, 'scheme','Department',1)}'s £{specific_value(gen_0_df, 'scheme','Scheme Value per year',1, decimal=1)} billion {specific_value(gen_0_df, 'scheme','Scheme Name',1)} and the {specific_value(gen_0_df, 'scheme','Department',2)}'s £{specific_value(gen_0_df, 'scheme','Scheme Value per year',2, decimal=1)} billion {specific_value(gen_0_df, 'scheme','Scheme Name',2)}."

In [ ]:
title = f"## Government grants statistics {year}\n\n"

In [ ]:
paragraph0 = (
    f"## Purpose of this report \n"
    f"This report accompanies the release of the Government grants register {yyy}."
    " This report provides context to this grants data, an overview of grant spending and guidance notes on"
    " how the data has been compiled.\n\n"
    "This release is classified as Official Statistics."
    " The details of the ongoing improvements to these statistics are outlined in the statistics development plan accompanying this publication.\n\n"
    f"Government grants register {yyy} data release covers:\n\n General grants and formula grants (at both scheme and award level)"
    " across all government departments."
    f"Note HM Treasury is not included in this publication because they do not manage any formula or general grants.\n * Grants funded"
    f" during the period 1 April {y1} to 31 March {y2}.\n * Grants that are new for {yyy} and grants that were set up in previous years which"
    " have continued to be active in {yyy}.\n * Exchequer-funded grants (this excludes grants funded by devolved governments of Scotland,"
    " Wales and Northern Ireland); this includes overseas aid provided by the UK government as grants, but excludes grants made by the EU"
    " or the UK government contribution to the EU.\n * Any central government departments or arm's length bodies (ALBs) that manage exchequer"
    " funded grants.\n\nThis data does not include:\n\n"
    "* Grants-in-aid - these are funds allocated from one part of government to another part of government, for example, central government"
    " funding for the running costs of non-departmental public bodies (NDPBs).\n"
    "* Details of awards made by Local Authorities.\n"
    "* Details of fraud and error. Data relating to fraud and error is reported centrally and published annually in the Cross-Government"
    " Fraud Landscape Annual Report.\n"
    "* Details of awards relating to the Coronavirus Job Recovery Scheme (CJRS) - information regarding these awards is published by HM"
    " Revenue and Customs (HMRC).\n\n"
    "There are a number of notes and caveats that will help inform the interpretation of this data. A summary of these, as well as"
    " departmental statements, can be found at the end of this report. Full details can be found in the Quality and Methodology Information"
    " document, published alongside this report."
    "\n Note that values presented in this report are rounded. Therefore totals may not equal the sum of their parts, and percentages may not"
    " add up to exactly 100%. The statistics in this report are based on unredacted data, so will not exactly match any statistics calculated"
    " directly from the accompanying government grants register.\n\n"
    "We would appreciate your feedback to help us improve future publications. This can be provided via this "
    "[feedback form](https://forms.gle/WMDCtS745sk3MZVX8)."
    f"\n\n## Introduction to government grants \n"
    f"Government grants are funds intended to be permanently"
	f"Government grants are funds intended to be permanently"
	f" transferred[^1] from a government organisation, to a grant recipient[^2], in order to fulfil a policy or"
	f" public interest need. Unlike contracts/procurements (e.g. for the purchase"
	f" of goods or services), grants are provided with the"
	f" focus on the outcomes and impacts of the activities"
	f" being funded. Usually grants are awarded to"
	f" finance (or reimburse expenditure on) the"
	f" recipients' activities in order to further the"
	f" implementation of government policy or public"
	f" interest, where it is neither appropriate nor possible"
	f" for the government organisation to carry out those"
	f" activities itself.\n\n"
	f"[^1]: unless unused or misspent\n"
	f"[^2]: a third party that is separate from the government organisation"
	f"\n\nGrant spending accounts for around {grant_percentage_of_expenditure}% of total UK government"
	f" expenditure.[^3] Government grant funding plays an important role domestically, in areas such as education,"
	f" research, civil society and innovation, and abroad through international aid projects."
	f"\n\n Grants can be used for a number of purposes, including providing financial subsidies to deliver activities and outcomes,"
	f" supporting government policy initiatives, and funding research, development and innovation."
	f" Examples of grant funding from 2021 to 2022 include awards to support government priorities including renewable energy both"
	f" domestically and abroad, fund local authorities to deliver new housing infrastructure, and deliver public health functions.\n\n"
	f"[^3]: Calculated based on the Office for Budget Responsibility's [Public finances databank](https://obr.uk/data/) figure for"
	f" total managed expenditure at the time of publication.\n\n"
	f"[^4]: Total COVID-19 spending is estimated by summing the total value of schemes that have been identified by departments as being"
	f" predominantly part of the response to the COVID-19 pandemic. Note there may be further schemes (not predominantly part of the COVID-19"
	f" response) that contained subsets of awards relating to COVID-19. Equally there may be schemes included in this estimate that would have"
	f" still paid out in the absence of the pandemic, or contain some spending that is not related to COVID-19.\n\n"
	f"During the financial year covered by this report, grant spending continued to play a significant role in the government's ongoing"
	f" response to the COVID-19 pandemic. Examples of this funding from {yyy} include the £8.3bn Self Employed Income Support Scheme (SEISS)"
	f" Extension and the £8.2bn CJRS from HMRC.\n \n"
	f"¶18\n\n"
	f"\n### Allocation of government grants"
	f"\nThere are two allocation methods by which grants are issued:"
	f"\n\n * **Formula grants:** are those calculated by way of a formula. This funding is provided, in"
	f" recognition of specific criteria, by central government to organisations such as local authorities,"
	f" schools and the police. Funding is determined by factors relevant to the purpose, such as population or"
	f" number of pupils who receive free school meals. These grants account for {percentage_of_total(formula_df)}%"
	f" (£{round(sum(formula_df['Scheme Value per year'])/1000000000,1)} billion across {len(formula_df)} schemes) of government grant"
	f" spending."
	f"\n\n * **General grants:** allow the government to secure policy objectives which the market cannot,"
	f" such as innovation and research, and they allow an effective funding route for the voluntary and"
	f" charitable sectors, for example to address homelessness and regional inequalities. General grants account for"
	f" {percentage_of_total(general_df)}% (£{round(sum(general_df['Scheme Value per year'])/1000000000,1)} billion across"
	f" {len(general_df)} schemes) of government grant spending.\n\n")

In [ ]:
paragraph1 = (
    "Source: Accompanying statistical tables - Table 1\n\n"
    f"###Total grant spend by department and allocation method\n\n"
    f"The government spent £{total_value_bn(scheme_df, 'scheme')} billion on grants in {yyy}. This is a 33% decrease from £258 billion "
    f"in 2020 to 2021. The "
    f"{specific_value(scheme_df, 'department', 'Department',1)} gave out the greatest amount"
    f" of money as grants, accounting for £{specific_value(scheme_df, 'department','Scheme Value per year',1,decimal=1)} billion "
    f"({percentage_value(scheme_df,1)}%) of the total value of government grants. {grants2}"
    f"{grants8}\n\n"
)

print(paragraph1)

In [ ]:
paragraph1_1 = (
    "Source: Accompanying statistical tables - Table 1\n\n"
    f"## How grant spending has changed over time - from Financial years 2018 to 2019 up to 2021 to 2022 \n"
    f"In the financial years prior to 2020 to 2021, grants spending was broadly static at £113 billion in 2018 to 2019, and £118 billion "
    f"in 2019 to 2020. This spend was dominated by formula grants schemes, making up around 70% of spend in both years.\n \n"
    f"In 2020 to 2021 and 2021 to 2022, grant spending played a key role in the UK government's response to the COVID-19 pandemic. Grant"
    f" spending in 2020 to 2021 was £258 billion, more than doubling compared to previous years, with general grant spending higher than"
    f" formula grant spending for the first time.\n\n"
    f"Grant spending in 2021 to 2022 has decreased by 33% compared to 2020 to 2021, however it remains significantly higher than in earlier"
    f" years. This is largely due to the continuation of grant spend to support the COVID-19 response with a value of £31.2 billion[^4], as well"
    f" as increased formula grant spending to support economic recovery."
    f" The breakdown of this spending has mostly returned to previous patterns, with formula grant spending making up"
    f" {percentage_of_total_grants_by_value(formula_df,scheme_or_award='scheme')}% of total grant spending.\n\n"
    f"The majority of formula grant spending continues to be associated with the DfE on schemes such as the DSG and the GAG."
    f" Grant spending by HMRC has reduced by 79% compared to 2020 to 2021 but is still significantly higher than in earlier years."
    f" This is due to the continuation of the CJRS and the SEISS. "
    f" Several other departments exhibit a similar trend to HMRC. Grant spending by the Department for Work and Pensions (DWP) increased by"
    f" 129% compared to 2020 to 2021. This was driven by the £744.8 million Kickstart scheme. Details on grant spending over time for all"
    f" departments can be found in Table 1 of the statistical tables accompanying this bulletin.\n\n"
)

In [ ]:
#Calculate % change in formula value compared to value last year
formula_change = abs(int(round((sum(formula_df['Scheme Value per year'])-110900000000)/110900000000*100)))

In [ ]:
paragraph2 = (
    f"Source: Accompanying statistical tables - Table 1\n\n"
    f"\n## Formula grants in financial year {yyy}\nFormula grants are those calculated by way of a formula. This funding is provided,"
    f" in recognition of specific criteria, by central government to organisations such as local authorities,"
    f" schools and the police. Funding is determined by factors relevant to the purpose, such as population"
    f" or number of pupils who receive free school meals. These grants account for"
    f" {percentage_of_total_grants_by_value(formula_df,scheme_or_award='scheme')}%"
    f" (£{total_value_bn(data=formula_df, award_or_scheme='scheme',decimal=1)} billion across {len(formula_df)} schemes)"
    f" of government grant spending. This is a {formula_change}% increase on the value in 2020 to 2021, when £110.9 billion was funded via"
    f" formula schemes.\n\n"
)

print(paragraph2)

In [ ]:
paragraph2_2 = (
    f"Source: Accompanying statistical tables - Table 1\n\n"
    "The largest formula grant funder"
    f" is the"
    f" {specific_value(formula_df,'department','Department',1)} who provided £"
    f"{specific_value(formula_df,'department','Scheme Value per year',1,decimal=1)} billion in formula grant funding during"
    f" {yyy}. {formula5} See table 1 below for details on these schemes, and the government grants register, accompanying this bulletin,"
    f" for details of all grant schemes.\n\n"
)

print(paragraph2_2)

In [ ]:
#Calculate % change in general value
general_change = abs(int(round((sum(general_df['Scheme Value per year'])-146900000000)/146900000000*100)))

In [ ]:
paragraph3 = (
    f"\n## General grants in financial year {yyy} \nGeneral grants allow the government to secure policy objectives which the market"
    " cannot, such as innovation and research, and they allow an effective funding route for the voluntary and"
    " charitable sectors, for example to address homelessness and regional inequalities. \n\nGeneral grants"
    f" account for {percentage_of_total_grants_by_value(general_df,scheme_or_award='scheme')}% (£{round(general_df['Scheme Value per year'].sum()/1000000000,1)} billion) of the value of government grants spending and"
    f" {percentage_of_total_grants_by_quantity(general_df)}% ({len(general_df)}) of the volume of grant schemes in {yyy}."
    f" This is a {general_change}% decrease on the value in 2020 to 2021, when £146.9 billion was funded via general grants. \n\n"
)

print(paragraph3)

In [ ]:
paragraph3_2 = (
    f"Source: Accompanying statistical tables - Table 1\n\n"
    f"The largest general grant funder in {yyy} was "
    f" {specific_value(general_df,'department','Department',1)} which spent £"
    f" the {specific_value(general_df,'department','Funder: Organisation Name',1)} ({specific_value(general_df,'department','Department',1)}) which spent £"
    f"{specific_value(general_df,'department','Scheme Value per year',1,decimal=1)} billion on general grants. {general2} {general5}"
    f" See table 2 below for details on these schemes, and the government grants register, accompanying this bulletin, for details of all"
    f" grant schemes.\n\n"
)

print(paragraph3_2)

In [ ]:
paragraph4 = (
    f"\n### General grant awards by allocation method in financial year {yyy}\n\n"
    f"There are three means by which general grants are allocated to recipients:\n\n * **Competed**"
    f" - applications are invited and evaluated, with awards made based on the outcome of the application."
    f" In {yyy} these grants had a total value of £{round(sum(competed_df['Total amount per year'])/1000000000,1)} billion. "
    f" Competed grant awards in {yyy} had an average (median)"
    f" value of £{int(round(competed_df['Total amount per year'].median()/1000,0))},000."
    f"As per the [Grants"
    f" Functional Standard](https://www.gov.uk/government/publications/grants-standards/grant-standards), competition should be the default"
    f" allocation method, wherever appropriate.\n\n"
    f" * **Un-competed** - grants are awarded to a single organisation or individual without a competition,"
    f" for example where there is only a single organisation that has the capability of delivering the"
    f" objectives. In {yyy} these grants had a total value of £{round(sum(uncompeted_df['Total amount per year'])/1000000000,1)} billion"
    f". Un-competed grant awards in {yyy} had an average (median)"
    f" value of £{int(round(uncompeted_df['Total amount per year'].median()/1000,0))},000. \n\n"
    f" * **Criteria** - disseminated based on specific qualifying criteria, for example grants to assist"
    f" those affected by floods. In {yyy} these grants had a total value of £"
    f"{round(sum(criteria_df['Total amount per year'])/1000000000,1)} billion."
    f" Criteria grant awards in {yyy} had an average (median) "
    f"value of £{int(round(criteria_df['Total amount per year'].median()/1000,0))},000.\n\n"
)

print(paragraph4)

In [ ]:
paragraph4_1 = (
    f"Source: Accompanying statistical tables - Table 2\n\n"
    f"Note that these totals are calculated based on award level data, and average (median) value are calculated based only on awards "
    f"to organisations. Therefore these totals do not add up to the other general grants totals in this report.\n\n"
)

In [ ]:
paragraph5 = (
    f"\n## Grant schemes by COFOG in financial year {yyy}\n\n"
    f"We can classify grants by their area of economic activity using the Classification of the functions"
    f" of government ([COFOG](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Glossary:Classification_of_the_functions_"
    f"of_government_(COFOG))). COFOG defines the broad objectives of government activity. The"
    f" UK government classifies £"
    f"{specific_value(data=scheme_df, by='cofog', column='Scheme Value per year',row=1, decimal=1)}"
    f" billion of its grants spending as "
    f"{specific_value(data=scheme_df, by='cofog', column='cofog', row=1)} ([see COFOG definitions](https://ec.europa.eu/eurostat/statistics"
    f"-explained/index.php?title=Glossary:Classification_of_the_functions_of_government_(COFOG)) for details on what areas of spending this"
    f" category includes)."
    f" {specific_value(data=scheme_df, by='cofog', column='cofog',row=2)} (£{specific_value(data=scheme_df, by='cofog', column='Scheme"
    f" Value per year',row=2, decimal=1)} billion) is the second largest classification.\n\n"
)

print(paragraph5)

In [ ]:
Paragraph6 = ""

In [ ]:
Paragraph7 = ""

In [ ]:
paragraph8 = (
    "Source: Accompanying statistical tables - Table 3\n\n"
    "\n### Data Notes\n"
    "Full notes for each data field are included in the scheme and award level data accompanying this "
    "report. Below are the key notes that users should be aware of when interpreting this data:"
    "\n\n * The location recorded on the register is not necessarily reflective of the ultimate beneficiary of "
    "grant funding. The address may represent the head office of the initial recipient, rather than "
    "where the money is actually spent."
    "\n * Some data has been redacted at both scheme and award level in the published data, where "
    "there is a requirement by law and data protection regulations; where data has been redacted at "
    "award level but not scheme level, this will result in a difference between scheme-level and "
    "award-level data values."
    "\n * The details of awards relating to the CJRS are excluded - information regarding these awards is published by HMRC."
    "\n * Grants-in-aid are excluded from this publication; grants-in-aid are funds allocated from one part "
    "of government to another part of government, for example, central government funding for the "
    "running costs of non-departmental public bodies (NDPBs)."
    "\n * Financial figures can be reported on either a cash or accruals basis. We are working with "
    "departments to make this consistent for future publications."
    "\n * Some data is completely redacted for reasons of national security or commercial security"
        "statistics in this report won't completely match the full published dataset."
    "\n * Some recipient information is redacted due to containing personal information or for national "
    "security or commercial security reasons. These details are replaced with [Redacted] in the "
    "dataset."
    "\n * Average (median) award values only include awards going to organisations, excluding those going to individuals, and are rounded to "
    "the nearest thousand."
    "\n * Where central government provides grant funds to an organisation that then provides onwards grants to end recipients, only the grant "
    "to the initial organisation (e.g. the local authority) is included in this data."
)

In [ ]:
paragraph9 = (
    "\n\n##Departmental statements about the data in this report\n"
    "\n\n**Cabinet Office**\n\n"
    "During the 2021-22 period, in addition to grants that would have been ordinarily issued by the Cabinet Office, of significant "
    "relevance, the Cabinet Office continued to support grant funding in relation to COP26 activities and the G7 summit. COP26, was the "
    "26th United Nations Climate Change conference, held at the SEC Centre in Glasgow, Scotland, United Kingdom, from 31 October to 13 "
    "November 2021 with support provided in the handover to COP27 held in Egypt. \n\nThe UK hosted the G7 Summit as part of its 2021G7 "
    "Presidency. The G7 Summit was held in Carbis Bay, Cornwall on 11-13 June 2021. In addition to this, following an announcement "
    "from the prime minister, a decision was made to provide additional grant funding to the office of veterans affairs to support "
    "veterans."

    "\n\n**Department for Business, Energy and Industrial Strategy**\n\n"
    "The BEIS data contained within the GGIS, and therefore within this publication, is an approximation of the grant schemes run by BEIS. "
    "The awards data covers grants to individual identifiable entities (companies, charities, universities) but excludes awards to persons "
    "and consortia. The data excludes non-grant award funding such as funding issued as contracts, loans and operating cost subsidies. "
    "Further details of these categories of spending are set out in the Department’s accounts and those of its partner organisations. "
    "The core BEIS grants data team is working with the Government Grants Management Function to improve data quality and completeness "
    "for the 2022 to 2023 publication. This is to strive for a better centralised representation of the grants delivered by BEIS, its "
    "partner organisations and ALBs. BEIS Covid support schemes allocated by local authorities are for eligible businesses in England. "
    "The Devolved Governments received funding for their own Covid support schemes which is not included in this report. \n\nFull details "
    "on the take up and the costs of the Help to Grow: Digital scheme will be released in due course once the scheme has fully closed."

    "\n\n**Department for Digital, Culture, Media and Sport**\n\n"
    "Please note that the total DCMS value reported in this submission is slightly above that reported in the Annual Report and Accounts as "
    "some grants report the budget value rather than the actual value and our data capture is not always capable of making this distinction."

    "\n\n**Department for Education**\n\n"
    "With the exception of the Office for Students (OfS), the DfE’s Government Grants Information System (GGIS) 2021-22 return for both "
    "grant schemes and grant awards is on an expenditure basis. This is consistent with treatment in prior years. DfE’s grant award data "
    "consists of one line for each recipient of the grant. The OfS’ grant schemes are reported on a budgeted basis, with awards being a "
    "mixture of budgeted and expenditure values. \n\nRespectively, the scheme data and award data reported on GGIS represent 97% and 94% "
    "of the Department Group’s grant expenditure. \n\nHaving been prepared on a resource expenditure basis, the total expenditure includes "
    "accounting adjustments (such as accruals) that are not relevant to users of the GGIS, along with transactions that we have been unable "
    "to attribute to specific recipients, and where further investigation was not proportionate on a value for money basis. These adjustments "
    "make up the 3% (scheme level) and 6% (award level) of total expenditure.\n\nWhen a grant has been awarded to an Academy, the Academy’s "
    "or its Academy Trust’s details may be present in the recipient organisation field. Where identified in the reporting process, the DfE’s "
    "return excludes funding to third party organisations through contracts."

    "\n\n**Department for Department for Environment, Food & Rural Affairs**\n\n"
    "During 2021-22 Defra has been focused on completing EU transition. We launched various marine and farming environment schemes including "
    "UK Seafood Fund, Future Farming Resilience Fund and Sustainable Farming Incentive to support this transition. \n\nWe lead on contributing "
    "to the Government’s 2050 net zero ambition and published the England Trees Action Plan and England Peatland Action Plan under the Nature "
    "for Climate Fund. Under this programme we launched a range of peatland restoration and tree planting schemes. \n\nWe launched the first "
    "set of programmes to receive funding from the £500 million Blue Planet Fund. This will increase marine protection, tackle plastic "
    "pollution and the decline of coral reefs, as well as supporting developing countries in nature-based solutions to tackle climate change "
    "and providing access to UK scientific expertise. We made record investment in flood and coastal defences, including providing grants to "
    "Local Authorities to help householders fund changes that will help them become more resilient to any future flooding. We continue to "
    "provide grant funding to Local Authorities to improve air quality, this year including grants focused on encouraging fleet turnover to "
    "newer, less polluting vehicles. For further details see Defra Annual Report and Accounts 2021-22. \n\nForests, woods and trees are at "
    "the heart of the Government’s agenda for tackling carbon and climate change, to improve the environment and to build the green economy. "
    "The target to treble tree planting rates in England is not without challenges and the associated risks are captured in the [Forestry "
    "Commission Annual Report](https://www.gov.uk/government/publications/forestry-commission-annual-report-and-accounts-2021-to-2022)."

    "\n\n**Department for Health and Social Care**\n\n"
    "The DHSC data covers core DHSC grants and those administered by its Arm’s Length Bodies. On 1st October 2021, Public Health England "
    "(PHE) was reconfigured to form the UK Health Security Agency (UKHSA) and part of the organisation was retained as a new directorate (OHID) "
    "within DHSC. The attribution of grants may therefore refer to DHSC, PHE or UKHSA, but the individual grants may have come under the remit of"

    "\n\n**Department for International Trade**\n\n"
    "DIT’s Annual Report and Accounts 2021-2022 includes grants funded by the EU Regional Development Fund. The data published in the governme"
        " those of these organisations during the financial year.\n\nDHSC data contained within the GGIS, and therefore within this "
    " publication, covers grants to entities such as companies, charities, universities and consortia but excludes awards to individuals. "
    " Funding issued as contracts for goods and services and as loans is also excluded, and we are working with Cabinet Office and Commercial "
    " Directorate to exclude partnership funding through Memoranda of Understanding for financial year 2022-23, which are not covered by the "
    " Government Grant Standard.\n\nPlease note that some of the data provided is budgeted rather than actual values and overall figures on the "
    " split between budgeted and actual values across government are included in the Quality and Methodology Information document which "
    " accompanies the publication.\n\nWhere the budget/actual figures for this year are listed as £0, this represents a no-cost extension, "
    " granted to allow grant recipients additional time to complete projects funded in the previous financial year.\n\nThe Disabled Facilities "
    " Grants scheme is funded by DHSC and administered by DLUHC on its behalf.\n\nEvery effort has been made to ensure that this data is "
    " accurate and only grant awards are included. However there may be a small number of awards listed that were ultimately funded through "
    " other funding mechanisms or not funded at all."

    "\n\n**Department for International Trade**\n\n"
    "DIT’s Annual Report and Accounts 2021-2022 includes grants funded by the EU Regional Development Fund. The data published in the "
    "government grants register includes only DIT grants funded by the UK Exchequer."

    "\n\n**Department for Levelling Up, Housing and Communities**\n\n"
    "The DLUHC data is on a cash payment basis for schemes and awards, which is consistent with prior years’ treatment, and therefore "
    "excludes accounting adjustments such as Accruals. \n\nGrants to local authorities include the Revenue Support Grant which finances "
    "revenue expenditure and capital grants which finance non-current assets. These are agreed through the local government finance "
    "settlement. In addition, specific grants are distributed outside the settlement. \n\nGrant payments may need to be recovered from "
    "recipients for a variety of reasons depending on the grant conditions. Where recoveries are made income is recognised at the point "
    "that the invoice, or other notice requiring repayment, has been issued. The negative values relate to grant recoveries which net off "
    "with the overall scheme at programme level. The Authority Acts for Awards are automatically associated to the relevant Scheme Authority "
    "Act on GGIS and may not be individually verified. A Scheme may have more than one relevant Authority Act. \n\nDLUHC continued to play a "
    "critical role in the government’s response to the COVID-19 pandemic through the year, with interventions across housing, local "
    "government and communities, to reduce transmission, protect the vulnerable and to help the country recover. To support local authorities "
    "with their ongoing COVID-19 Response role, the Government allocated over £12 billion directly to councils in 2020-21 and 2021-22. "
    "Further information on Covid expenditure can be found in the department’s Annual Report and Accounts 2021/22. \n\nNew UK-wide growth "
    "funds begun delivery this year, including the Levelling Up Fund and UK Community Renewal Fund, funding projects across the UK. \n\nIn "
    "October 2021 the department uncovered a control failing which allowed the GLA to accumulate £1.7 billion of funding from the 2016-23 "
    "Affordable Homes Programme that they had not spent on delivery since programme payments began in 2015-16. The surplus funding had been "
    "correctly set aside for investment in affordable housing. DLUHC officials took immediate action to stop further payments to the GLA and "
    "to recover payments made in 2021-22. Therefore after recoveries, the 2016-2023 Affordable Housing Programme - London scheme had zero "
    "net funding in 2021 to 2022 and so doesn’t appear in this publication. Further information on the Affordable Homes Programme can be "
    "found in the department’s Annual Report and Accounts 2021/22. \n\nBusiness rates retention (top ups) are excluded from this publication "
    "as these are redistributed business rate grant payments funded through the collection of tariff amounts under the rate retention system "
    "and are not funded via the UK Exchequer. \n\nThe Disabled Facilities Grant is funded by DHSC and is therefore included in their return "
    "this year."

    "\n\n**Department for Transport**\n\n"
    "The award amounts for ‘Covid 19 Bus Service Support Grant commercial’ have been redacted and appear as £0 in this publication. Specific "
    "amounts of grants paid to each company could risk distorting competition in the sector."

    "\n\n**Foreign, Commonwealth & Development Office**\n\n"
    "The data contained within this Government Grants Register shows a single year snapshot of FCDO grant spend as it is held on the GGIS at "
    "the point of publication.\n\nThe most accurate and up to date information on FCDO programmes can be found under [FCDO transparency "
    "releases](https://www.gov.uk/search/transparency-and-freedom-of-information-releases?parent=foreign-commonwealth-development-office&"
    "content_store_document_type%5B%5D=transparency&organisations%5B%5D=foreign-commonwealth-development-office&order=updated-newest) on "
    "gov.uk. All Overseas Development Assistance (ODA), including FCDO grant funding, is published to the International Aid Transparency "
    "Initiative (IATI) standard on [Development Tracker](https://devtracker.fcdo.gov.uk/)."

    "\n\n**HM Revenue & Customs**\n\n"
    "Throughout 2021-22, HMRC’s grants were primarily made up of COVID-19 scheme payments - specifically, the SEISS and the CJRS. The "
    "figures represent actual expenditure, as published in our 2021-22 Annual Report and Accounts. The 2021-22 values are substantially "
    "lower than 2020-21 totals due to COVID schemes coming to a close during the financial year."

    "\n\n**Home Office**\n\n"
    "The HO data contained within the GGIS, and therefore within this publication, is reflective of the grant schemes run by HO.\n\nThe "
    "awards data covers grants to individual identifiable entities (companies, charities, universities) and other bodies. The data excludes "
    "non-grant award funding such as funding issued as contracts.\n\nThe figures represent either budgeted or actual expenditure, depending "
    "on the information available up to the time of completion and additional expenditure may have occurred since these details were "
    "presented and accounts for variances between the scheme value and the supporting awards. Any schemes with £0 value had a break in "
    "continuity for the funding year being published.\n\nThe core HO grants team is working with the Government Grants Management Function "
    "to improve data quality and completeness.\n\nSome scheme details are not included in the publication report due to their sensitive "
    "nature or for reasons of national security."

    "\n\n**Ministry of Defence**\n\n"
    "There are some differences between the financial year value of some schemes and the total financial year value of the associated awards. "
    "These variations exist due to several factors; COVID-19 restrictions prevented some commemorative events taking place; challenges around "
    "recruitment; awards being considered and approved by award panels in the final quarter of the Financial Year and a headcount variation/"
    "basis to which these awards are made."

    "\n\n**Ministry of Justice**\n\n"
    "Addresses of Rape Support Funding Grant recipients and Women’s Community Sector Grant recipients have been redacted as they contain "
    "sensitive information."
)

In [ ]:
#The cell reads in the departmental Statements workbook then puts those departments and statements into the correct format for the Gov speak text.
#Within the statements if there should be spaces between paragraphs make sure these are done within the excel cells and they will be read through.

statements = pd.read_excel(r'..\data\Departmental Statements.xlsx', 0)
statements = statements.sort_values(by='Department', ascending=True)

paragraph9 = f"\n\n##Departmental statements about the data in this report\n"

for index, row in statements.iterrows():
    paragraph9 += f"\n\n**{row['Department']}**\n\n"
    #Puts dept name in right format for heading
    paragraph9 += f"{row['Statement']}\n"
    #Puts statement in correct format

print(paragraph9)

In [ ]:
#Folder to save the figures
blended = 'Blended'

if not os.path.exists(folder+'/' + blended):
    os.makedirs(folder+'/' + blended)

In [ ]:
from matplotlib import rc

#rc('font', **{'family':'sans-serif','sans-serif':['Helvetica']})
rc('font', **{'family':'sans-serif','sans-serif':['Arial']})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), subplot_kw=dict(aspect="equal"))

labels = [
    'Formula\n£' + str(round(sum(formula_df['Scheme Value per year']) / 1000000000, 1)) + ' billion\n' +
    str(int(round(sum(formula_df['Scheme Value per year']) / sum(scheme_df['Scheme Value per year']) * 100, 0))) + '%',

    'General\n£' + str(round(sum(general_df['Scheme Value per year']) / 1000000000, 1)) + ' billion\n' +
    str(int(round(sum(general_df['Scheme Value per year']) / sum(scheme_df['Scheme Value per year']) * 100, 0))) + '%'
]

data = [
    sum(formula_df['Scheme Value per year']),
    sum(general_df['Scheme Value per year'])
]

wedges, texts = ax.pie(
    data,
    colors=['#f47738', '#1d70b8'],
    wedgeprops={'linewidth': 3, 'edgecolor': 'white'},
    startangle=90
)

bbox_props = dict(boxstyle="square,pad=0.3", fc="w", ec="k", lw=0.72)
kw = dict(
    arrowprops=dict(arrowstyle="-"),
    bbox=bbox_props,
    zorder=0,
    va="center"
)

my_circle = plt.Circle((0, 0), 0.6, color='white')
#plt.gcf()
p.gca().add_artist(my_circle)

for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1) / 2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))
    horizontalalignment = {-1: "center", 1: "center"}[int(np.sign(x))]
    connectionstyle = "angle,angleA=0,angleB={}".format(ang)
    kw["arrowprops"].update({"connectionstyle": connectionstyle})

    ax.annotate(
        labels[i],
        xy=(x, y),
        xytext=(1.6*np.sign(x), 0.05),
        xycoords='data',
        textcoords='data',
        va='center',
        horizontalalignment=horizontalalignment,
        fontsize=13
    )

ax.set_title(
    "Figure 1: Proportion of grant spending by scheme allocation method",
    fontsize=13
)

set_size(2.5, 2.5)

plt.savefig(
    folder + '/' + blended + '/' + today + 'Figure1.svg',
    bbox_inches='tight'
)

In [ ]:
plt.savefig(folder + '/' + blended + '/' + today + 'Figure1.svg', bbox_inches='tight')
plt.show()

In [ ]:
table1 = chart(
    department_or_scheme='department',
    stacked=True,
    data1=formula_df,
    data2=general_df,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 1: Total grant spending by department and allocation method',
    excel_sheet_name='Table 1',
    table_name='1 Grant spending by department and scheme allocation method'
)

table_1 = chart(
    svg=True,
    department_or_scheme='department',
    stacked=True,
    data1=formula_df,
    data2=general_df,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 2: Total grant spending by department and allocation method',
    file_name='Figure2.svg'
)

In [ ]:
historical_data = read_csv_clean(
    r'..\data\historical_values.csv',
    dtype='object'
)

current_data = pd.DataFrame({
    'Year': ["2023/24"],
    'General': [sum(general_s_df['Scheme Value per year'])],
    'Formula': [sum(formula_s_df['Scheme Value per year'])]
})

over_time_data = pd.concat([current_data, historical_data])

over_time_data['General'] = pd.to_numeric(over_time_data['General'])
over_time_data['Formula'] = pd.to_numeric(over_time_data['Formula'])
over_time_data['Total'] = over_time_data['General'] + over_time_data['Formula']

over_time_data


In [ ]:
width = 0.8

fig, ax = plt.subplots(frameon=False)

data = over_time_data.sort_values(
    by=['Year'],
    ascending=True
).reset_index(drop=True)

labels = data['Year']
column1 = data['Formula']
column2 = data['General']

print(column1)
print(column2)

ax.bar(
    labels,
    column1,
    width,
    label='Formula',
    color='#f47738'
)

ax.bar(
    labels,
    column2,
    width,
    bottom=column1,
    color='#1d70b8',
    label='General'
)

ax.set_frame_on(False)  # use true/false to show/hide border
ax.get_yaxis().set_visible(False)

ax.legend(
    loc='upper left',
    bbox_to_anchor=(.0, 1.0),
    ncol=1,
    fontsize=15
)  # use bbox_to_anchor to change position of legend

plt.xticks(fontsize=12)

plt.ylim(
    0,
    sum(scheme_df['Scheme Value per year']) + 120000000000
)  # use the added number to increase y scale of graph

#set_size(5,len(data)/2.5)

size_font = 10
#ax.set_ylabel('Total spending', fontsize=size_font)

for i, v in enumerate(data['Total']):
    plt.text(
        i - 0.43,
        v + 7000000000,
        '£' + str(round(v/1000000000)) + " billion",
        color='black',
        fontsize=size_font
    )

# use 'i-0' to move value labels across top of bars

title = "Figure 3: Total grant spending by financial year and allocation method"

ax.set_title(title, fontsize=size_font)
plt.yticks(fontsize=size_font)

plt.savefig(
    folder + '/' + blended + '/' + today + 'Figure3.svg',
    bbox_inches='tight'
)

plt.show()

In [ ]:
table2 = chart(
    department_or_scheme='department',
    stacked=False,
    data1=formula_df,
    data2=None,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 4: Formula grant spend by department'
)

table_2 = chart(
    svg=True,
    department_or_scheme='department',
    stacked=False,
    data1=formula_df,
    data2=None,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 4: Formula grant spend by department',
    file_name='Figure4.svg',
    formula_or_general='formula'
)

In [ ]:
table3 = chart(
    department_or_scheme='scheme',
    stacked=False,
    data1=formula_df,
    data2=None,
    merged_field=None,
    field2='Value',
    rows=10,
    title='Table 1: The 10 largest formula grants schemes'
)

table_3 = scheme_tables(formula_df, rows=10)

table3 = table3.replace('\n', ' ').replace('\r', '')

#Note - line break characters in scheme aims & objectives may mess up the formatting of the table in the output
#So you need to remove these on GGIS or just in the 'Rewritten Purpose and objectives' column of the top 10 schemes file

In [ ]:
table4 = chart(
    department_or_scheme='department',
    stacked=False,
    data1=general_df,
    data2=None,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 5: General grant spend by department'
)

table_4 = chart(
    svg=True,
    department_or_scheme='department',
    stacked=False,
    data1=general_df,
    data2=None,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 5: General grant spend by department',
    file_name='Figure5.svg',
    formula_or_general='general'
)

In [ ]:
table6 = chart(
    department_or_scheme='allocation_method',
    stacked=False,
    data1=organisation_general_awards,
    data2=None,
    title='Figure 6: Median value by allocation method',
    excel_sheet_name='Table 2',
    allocation_method='median'
)

In [ ]:
table5 = chart(
    department_or_scheme='allocation_method',
    stacked=False,
    data1=general_a_df,
    data2=None,
    title='Figure 6: Value by allocation method',
    excel_sheet_name='Table 2',
    allocation_method='sum',
    table_name='2 Average and total value of general grant awards by general allocation method'
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), subplot_kw=dict(aspect="equal"))

labels = (
    'Criteria\n£' +
    str(round(sum(criteria_df['Total amount per year']) / 1000000000, 1)) +
    ' billion\n' +
    str(int(round(
        sum(criteria_df['Total amount per year']) /
        sum(general_a_df['Total amount per year']) * 100,
        0
    ))) + '%',

    'Competed\n£' +
    str(round(sum(competed_df['Total amount per year']) / 1000000000, 1)) +
    ' billion\n' +
    str(int(round(
        sum(competed_df['Total amount per year']) /
        sum(general_a_df['Total amount per year']) * 100,
        0
    ))) + '%',

    'Un-competed\n£' +
    str(round(sum(uncompeted_df['Total amount per year']) / 1000000000, 1)) +
    ' billion\n' +
    str(int(round(
        sum(uncompeted_df['Total amount per year']) /
        sum(general_a_df['Total amount per year']) * 100,
        0
    ))) + '%'
)

data = [
    sum(criteria_df['Total amount per year']),
    sum(competed_df['Total amount per year']),
    sum(uncompeted_df['Total amount per year'])
]

wedges, texts = ax.pie(
    data,
    colors=['#1d70b8', '#003078', '#5694ca'],
    wedgeprops={'linewidth': 3, 'edgecolor': 'white'},
    startangle=90
)


bbox_props = dict(boxstyle="square,pad=0.3", fc="w", ec="k", lw=0.72)
kw = dict(arrowprops=dict(arrowstyle="-"), bbox=bbox_props, zorder=0, va="center")

my_circle = plt.Circle((0,0), 0.6, color='white')
p = plt.gcf()
p.gca().add_artist(my_circle)

for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1)/2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))
    horizontalalignment = {-1: "center", 1: "center"}[int(np.sign(x))]
    connectionstyle = "angle,angleA=0,angleB={}".format(ang)
    kw["arrowprops"].update({"connectionstyle": connectionstyle})

    ax.annotate(
        labels[i],
        xy=(x, y),
        xytext=(1.55*np.sign(x)+.05, 1*y-.25),
        horizontalalignment=horizontalalignment,
        fontsize=13
    )

set_size(3,3)

ax.set_title(
    "Figure 6: Total value of general awards by general allocation method",
    fontsize=13
)

plt.savefig(
    folder + '/' + blended + '/' + today + 'Figure6.svg',
    bbox_inches='tight'
)

plt.show()

In [ ]:
#Need to fix this

table7 = chart(
    department_or_scheme='cofog',
    stacked=False,
    data1=scheme_df,
    data2=None,
    merged_field='cofog',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 7: Value of schemes by economic classification (COFOG)',
    excel_sheet_name='Table 3',
    table_name='3 Grant spend by economic classification (COFOG)'
)

table_7 = chart(
    svg=True,
    department_or_scheme='cofog',
    stacked=False,
    data1=scheme_df,
    data2=None,
    merged_field='cofog',
    field2='Scheme Value per year',
    rows=10,
    title='Figure 7: Value of schemes by economic classification (COFOG)',
    file_name='Figure7.svg'
)

In [ ]:
print(general_df['Department'].value_counts())

In [ ]:
table8 = chart(
    department_or_scheme='scheme',
    stacked=False,
    data1=general_df,
    data2=None,
    merged_field='Department',
    field2='Scheme Value per year',
    rows=10,
    title='Table 2: The 10 largest general grants schemes'
)

table_8 = scheme_tables(general_df, rows=10)

table8

In [ ]:
#Note - line break characters in scheme aims & objectives may mess up the formatting of the table in the output
#So you need to remove these on GGIS or just in the 'Rewritten Purpose and objectives' column of the top 10 schemes file

In [ ]:
workbook = writer.book
worksheet = workbook.add_worksheet('Notes')

# Add A1
A1 = workbook.add_format({'bold': True, 'font_size':24})
worksheet.write('A1', 'Notes', A1)

#Add A3
worksheet.write(
    'A3',
    'The statistics in these tables are based on unredacted data, so will not exactly match any statistics calculated from the government grants register accompanying these tables.'
)

worksheet.write(
    'A4',
    'Scheme and award values might not match as scheme level data is based on budgets of the whole scheme and award level data is based on the budgets or actual payments of individual awards.'
)

#Add A7 and B7
header_format2 = workbook.add_format({
    'bold': True,
    'font_size': 11,
    'bottom': True,
    'top': True
})

worksheet.write('A7', 'Acronym', header_format2)
worksheet.write('B7', 'Department', header_format2)

row = 7

for i in range(len(department_acronym)):
    for j in range(department_acronym.shape[1]):
        if j == 0:
            worksheet.write(row+i, j+1, department_acronym.values[i][j])
        elif j == 1:
            worksheet.write(row+i, j-1, department_acronym.values[i][j])

worksheet.set_column('B:B', 50)

In [ ]:
govspeak = 'Govspeak_ONLY'

if not os.path.exists(folder+'/' +govspeak):
    os.makedirs(folder+'/' +govspeak)

path1 = folder + '/' + govspeak + '/' + today + ' Government grants statistical tables ' + yy + '.txt'  #name of .txt file that will be exported

space = "\n\n"

print(type(table3))

gov_speak_only = paragraph0 + paragraph1 + table1 + paragraph2 + table2 + paragraph2_2 + table3 + space + paragraph3 + table4 + paragraph3_2 + paragraph4 + table5 + space + paragraph5 + table7 + space
gov_speak_only = gov_speak_only + table8
gov_speak_only = gov_speak_only.replace("\x93", "")
gov_speak_only = gov_speak_only.replace("\x94", "")

with open(path1, 'w', encoding='utf-8') as _file:
    _file.write(gov_speak_only)

In [ ]:
path1 = folder + '/' + blended + '/' + today + '_Statistics_Bulletin_' + yy + '.txt'  #name of .txt file that will be exported

space = "\n\n"

full_gov_speak = paragraph0 + '!!1 \n\n' + paragraph1 + '!!2 \n\n' + paragraph1_1 + '!!3 \n\n' + paragraph2 + '!!4 \n\n' + paragraph2_2 + table3 + "\n\n" + paragraph3 + '!!5\n' + paragraph3_2 + table8 + "\n\n" + paragraph4 + '!!6\n\n' + paragraph4_1 + paragraph5 + '!!7\n\n' + paragraph8 + paragraph9

full_gov_speak = full_gov_speak.replace("\x93", "")
full_gov_speak = full_gov_speak.replace("\x94", "")
full_gov_speak = full_gov_speak.replace("\u2011", "-")

with open(path1, 'w', encoding='utf-8') as _file:
    _file.write(full_gov_speak)

In [ ]:
path2 = folder + '/' + blended + '/' + today + '_Statistics_Bulletin_key_statistics_' + yy + '.txt'  #name of .txt file that will be exported

space = "\n\n"

with open(path2, 'w', encoding='utf-8') as _file:
    _file.write(
        f"total grant spending - {total_grant_value}\n"
        f"percentage of government expenditure {grant_percentage_of_expenditure}\n"
        f"number of schemes- {len(scheme_df)}\n"
        f"number of awards- {len(organisation_awards)}\n\n"
        f"formula schemes - {len(formula_df)}\n"
        f"formula awards- {len(organisation_formula_awards)}\n"
        f"general schemes- {len(general_df)}\n"
        f"general awards- {len(organisation_general_awards)}\n\n"
        f"total formula spend- {formula_df['Scheme Value per year'].sum()}\n\n"
        f"median formula award {median_value(organisation_formula_awards, award=True)}\n\n"
        f"total general spend- {general_df['Scheme Value per year'].sum()}\n\n"
        f"median general award {median_value(organisation_general_awards, award=True)}"
    )

In [ ]:
#This cell reads in the image 'Key Statistics' then prints numerical values onto the image. If the input should change upload a new 'blank'
#image to the data folder. Note that changes in the image may mean text printed onto the image is out of line so this may need adjusted below.
#Similarly if figures change by an order of magnitude this could cause misalignment.

In [ ]:
#This cell reads in the image 'Key Statistics' then prints numerical values onto the image. If the input should change upload a new 'blank'
#image to the data folder. Note that changes in the image may mean text printed onto the image is out of line so this may need adjusted below.
#Similarly if figures change by an order of magnitude this could cause misalignment.

img = Image.open(r'..\data\Key Statistics.png')  #Read in the image
draw = ImageDraw.Draw(img)  #Open in ImageDraw

# Total spend stat
font = ImageFont.truetype("arialbd.ttf",30)
#                 Choose font and size
txt = fr"£{total_grant_value} billion"
#                 Write text
draw.text((135, 95), txt, fill=(0, 0, 0), font=font)  # Set text position, color and font

# Repeat for all stats

# % expenditure stat
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{grant_percentage_of_expenditure}%"

draw.text((435, 95), txt, fill=(0, 0, 0), font=font)

# Number of schemes
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{len(scheme_df):,}"
draw.text((322, 190), txt, fill=(0, 0, 0), font=font)

# Number of awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{len(organisation_awards):,}"
draw.text((510, 190), txt, fill=(0, 0, 0), font=font)

# Number of formula schemes
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{len(formula_df):,}"
draw.text((40, 355), txt, fill=(255, 255, 255), font=font)

# Number of formula awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{len(organisation_formula_awards):,}"
draw.text((300, 355), txt, fill=(255, 255, 255), font=font)

# Number general schemes
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{len(general_df):,}"
draw.text((440, 355), txt, fill=(255, 255, 255), font=font)

# Number general awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"{len(organisation_general_awards):,}"
draw.text((720, 355), txt, fill=(255, 255, 255), font=font)

# Value of formula awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"£{round(formula_df['Scheme Value per year'].sum()/10**9,1):,} billion"

draw.text((45, 415), txt, fill=(255, 255, 255), font=font)

# Median value of formula awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"£{round(median_value_numerical(organisation_formula_awards, award=True)*1000):,}"
print(txt)
draw.text((300, 415), txt, fill=(255, 255, 255), font=ImageFont.truetype("arialbd.ttf",30))

In [ ]:
draw.text((300, 415), txt, fill=(255, 255, 255), font=ImageFont.truetype("arialbd.ttf",30))

# Value of general awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"£{round(general_df['Scheme Value per year'].sum()/(10**9),1):,} billion"
draw.text((440, 415), txt, fill=(255, 255, 255), font=font)

# Median value of general awards
font = ImageFont.truetype("arialbd.ttf",30)
txt = fr"£{round(median_value_numerical(organisation_general_awards, award=True)*1000):,}"
draw.text((720, 415), txt, fill=(255, 255, 255), font=font)

img.save(f"../data/{today}_Statistics_Bulletin_script_exports/blended/{today}_Key_Statistics_{yyy}.png")

# Uncomment below to show image when cell is run
img.show()

In [ ]:
writer.save()
writer.close()

In [ ]:
#Folder to save the figures
signoff = 'Signoff files'
if not os.path.exists(folder+'/' +signoff):
    os.makedirs(folder+'/' +signoff)

In [ ]:
path_formula = folder + '/' + signoff + '/' + today + '-formula_df.csv'
path_general = folder + '/' + signoff + '/' + today + '-general_df.csv'

formula_df.to_csv(path_formula, index=False, encoding=CSV_OUT_ENCODING)
general_df.to_csv(path_general, index=False, encoding=CSV_OUT_ENCODING)

In [ ]:
def departments_by_value(data):
    #function that orders departments by value
    #data - the data set that is to be analysed
    data = data.groupby(
        ['Department', 'Funder: Organisation Name']
    )[['Scheme Value per year']].agg('sum').sort_values(
        by=['Scheme Value per year'],
        ascending=False
    ).reset_index()

    data['Scheme Value per year'] = data['Scheme Value per year'].astype(float)

    return data

In [ ]:
folder = f"../data/" + today + '_Files for department signoff'
if not os.path.exists(folder):
    os.makedirs(folder)

In [ ]:
#Static Files
department_acronym = read_csv_clean(
    r'..\data\2021-01-01-DepartmentAcronyms.csv',
    dtype='object'
)

#Dynamic Files
# Paths with dynamic data based on today. These CSVs are written as UTF-8-SIG above.
formula_path = '../data/' + f"{today}_Statistics_Bulletin_script_exports/Signoff files/" + f"{today}" + '-formula_df.csv'
general_path = '../data/' + f"{today}_Statistics_Bulletin_script_exports/Signoff files/" + f"{today}" + '-general_df.csv'
scheme_path = '../data/' + f"{today}_Government_Grants_Register_exports/Signoff files/" + f"{today}" + '-scheme_register_df.csv'
award_path = '../data/' + f"{today}_Government_Grants_Register_exports/Signoff files/" + f"{today}" + '-award_register_df.csv'

formula_df = read_csv_clean(formula_path, dtype='object')


In [ ]:
general_df = read_csv_clean(general_path, dtype='object')


In [ ]:
scheme_register = read_csv_clean(scheme_path, dtype='object')


In [ ]:
award_register = read_csv_clean(award_path, dtype='object')


In [ ]:
#Adding in scheme purpose into scheme dataset, so this can be included in sign off file (field won't be published)

#Uncomment the following line to use manually downloaded report
#SchemePurposes = read_csv_clean(r'..\data//' + today + ' Scheme Purpose Report.csv', dtype='object')

#merge with scheme dataset on scheme ref:
print(len(scheme_register))
scheme_register = scheme_register.merge(
    scheme_purposes,
    left_on='Grant Programme:Code',
    right_on='Scheme Reference #',
    how='left'
)

print(len(scheme_register))

#Convert scheme purpose to 'covid' or 'not covid' only (for publication we just need to calculate a total covid figure)
scheme_register['Scheme purpose'] = np.where(
    scheme_register['Scheme purpose'] == 'COVID-19',
    'COVID-19',
    'Not COVID'
)

In [ ]:
#Function utilised to create data set for scheme total values by department broken down by allocation method

def departments_by_value(data):
    #function that orders departments by value
    #data - the data set that is to be analysed
    data = data.groupby(
        ['Department', 'Funder: Organisation Name']
    )[['Scheme Value per year']].agg('sum').sort_values(
        by=['Scheme Value per year'],
        ascending=False
    ).reset_index()

    data['Scheme Value per year'] = data['Scheme Value per year'].astype(float)

    return data

In [ ]:
#merge formula and general data sets where column 0 is Department, column 1 is formula scheme value and column 2 is General scheme value
stats_bulletin = pd.merge(
    departments_by_value(formula_df)[['Department', 'Scheme Value per year']],
    departments_by_value(general_df)[['Department', 'Scheme Value per year']],
    on='Department',
    how='outer'
).fillna(0)

#create 4th column representing the total scheme value for each department
stats_bulletin['Total'] = stats_bulletin['Scheme Value per year' + '_x'] + stats_bulletin['Scheme Value per year' + '_y']

#rename, change data type and order in descending order by total value
stats_bulletin = stats_bulletin.rename(columns={
    'Scheme Value per year' + '_x': 'Formula',
    'Scheme Value per year' + '_y': 'General'
})

stats_bulletin['Formula'] = stats_bulletin['Formula'].astype(float)
stats_bulletin['General'] = stats_bulletin['General'].astype(float)
stats_bulletin['Total'] = stats_bulletin['Total'].astype(float)

stats_bulletin = stats_bulletin.sort_values(
    by=['Total'],
    ascending=False
).reset_index(drop=True)

In [ ]:
############################################################################################

#Steps taken to create a table of the 20 schemes listed in the top ten schemes in the stats bulletin

#Formula Data
#Change column name and filter for do not publish
stats_bulletin_formula = formula_df
stats_bulletin_formula['Value'] = stats_bulletin_formula['Scheme Value per year']
stats_bulletin_formula = stats_bulletin_formula[
    stats_bulletin_formula['Do not publish'] != 1.0
]

#Select columns needed for excel and create additional column for excel
stats_bulletin_formula = stats_bulletin_formula[[
    'Department',
    'Scheme Name',
    'Purpose and objectives',
    'Value'
]].sort_values(
    by=['Value'],
    ascending=False
).head(10).reset_index(drop=True)

stats_bulletin_formula = stats_bulletin_formula[[
    'Department',
    'Scheme Name',
    'Purpose and objectives',
    'Value'
]].reset_index(drop=True)

stats_bulletin_formula['Allocation method'] = 'Formula'

In [ ]:
#General data
#Change column name and filter for do not publish
stats_bulletin_general = general_df
stats_bulletin_general['Value'] = stats_bulletin_general['Scheme Value per year']
stats_bulletin_general = stats_bulletin_general[
    stats_bulletin_general['Do not publish'] != 1.0
]

#Select columns needed for excel adn create additional column for excel
stats_bulletin_general = stats_bulletin_general[[
    'Department',
    'Scheme Name',
    'Purpose and objectives',
    'Value'
]].sort_values(
    by=['Value'],
    ascending=False
).head(10).reset_index(drop=True)

stats_bulletin_general = stats_bulletin_general[[
    'Department',
    'Scheme Name',
    'Purpose and objectives',
    'Value'
]].reset_index(drop=True)

stats_bulletin_general['Allocation method'] = 'General'

#Merge the two lists together
largest_schemes = pd.concat(
    [stats_bulletin_formula, stats_bulletin_general],
    ignore_index=True
)

In [ ]:
#comment out line below if you want to run the script on over all departments
#department_acronym = department_acronym[department_acronym['Acronym']==Department_to_be_exported].reset_index(drop=True)

In [ ]:
for a in range(len(department_acronym)):
    i = department_acronym.at[a, 'Acronym']  # iterate over acronym list, where i = department acronym eg 'BEIS'

    path0 = f"{folder}/{today} {i}_GGIS_{y}_signoff.xlsx"  #name of excel that will be exported

    writer = pd.ExcelWriter(
        path0,
        engine='xlsxwriter'
    )  # opening function that writes data to the open excel

    workbook = writer.book

    #cell formatting utilised through excel file
    heading = workbook.add_format({
        'bold': True,
        'font_size': 18
    })

    heading2 = workbook.add_format({
        'bold': True,
        'font_size': 14
    })

    wraptext = workbook.add_format({
        'text_wrap': True
    })

    header = workbook.add_format({
        'bg_color': '#4f81bd',
        'bold': True,
        'bottom': True,
        'top': True,
        'left': True,
        'right': True
    })

    text_dark = workbook.add_format({
        'bg_color': '#b8cce4',
        'bottom': True,
        'top': True,
        'left': True,
        'right': True
    })

    text_light = workbook.add_format({
        'bg_color': '#dbe5f1',
        'bottom': True,
        'top': True,
        'left': True,
        'right': True
    })

    numerical_dark = workbook.add_format({
        'num_format': '#,##0_-;[Red]-#,##0_-;_-',
        'bg_color': '#b8cce4',
        'bottom': True,
        'top': True,
        'left': True,
        'right': True
    })

    numerical_light = workbook.add_format({
        'num_format': '#,##0_-;[Red]-#,##0_-;_-',
        'bg_color': '#dbe5f1',
        'bottom': True,
        'top': True,
        'left': True,
        'right': True
    })

    currency_format = workbook.add_format({
        'italic': True,
        'font_size': 11,
        'align': 'right'
    })

    #Creating first sheet - 'Contents'
    worksheet = workbook.add_worksheet('Contents')  # Create first sheet of excel

    #Set column width in sheet
    worksheet.set_column('A:A', 25)
    worksheet.set_column('B:B', 60)
    worksheet.set_column('C:C', 50)

    #Input text cell-by-cell
    # Add A1
    worksheet.write(
        'A1',
        f"{i} {yyy} GGIS signoff",
        heading
    )

    #Add A3
    worksheet.write(
        'A3',
        f"As per the instructions in the email to which this file was attached, please sign-off the data and statistics in this file by {signoff_date}."
    )

    #Add A5, B5 & C5
    worksheet.write('A5', 'Tab', heading2)
    worksheet.write('B5', 'Description', heading2)
    worksheet.write('C5', 'Actions', heading2)

    #Add A6, A7 & A8
    worksheet.write('A6', 'Statistical report figures')
    worksheet.write('A7', 'Schemes to publish')
    worksheet.write('A8', 'Awards to publish')

In [ ]:
#Add B6, B7 & B8
worksheet.write(
    'B6',
    'Contains those figures for your department that will be published in the statistical report. (these figures include data from "Do not publish" grants so may not match the scheme or awards data)',
    wraptext
)
worksheet.write(
    'B7',
    'A table of scheme level data that we will be publishing (redacted where labelled as "Do not publish" on GGIS)',
    wraptext
)
worksheet.write(
    'B8',
    'A table of award level data that we will be publishing (redacted where labelled as "Do not publish" on GGIS)',
    wraptext
)

#Add C6, C7 & C8
worksheet.write(
    'C6',
    '> Check that these figures are correct\n> Provide us with updated names, descriptions and outcomes of any large schemes listed in this tab',
    wraptext
)
worksheet.write(
    'C7',
    '> Check that this data is correct\n> Highlight any data that should be redacted\n> Ensure that this data contains no personal information',
    wraptext
)
worksheet.write(
    'C8',
    '> Check that this data is correct\n> Highlight any data that should be redacted\n> Ensure that this data contains no personal information',
    wraptext
)

#Add A10
worksheet.write('A10', f"Scope of the data we will publish", heading)

#Add A11
worksheet.write('A11', f"We will publish any approved schemes or awards that are:")

#Add A12,A13, A14 & A15
worksheet.write('A12', f"      Funded by the UK Exchequer")
worksheet.write('A13', f'      Not labelled as "Do not publish" on GGIS')
worksheet.write('A14', f"      Active in {yyy}")
worksheet.write('A15', f"      Not grants in aid")

In [ ]:
####################################################################################################
#Creating second sheet - 'Statistical report figures'
worksheet = workbook.add_worksheet('Statistical report figures')

In [ ]:
#Total value of scheme expenditure by allocation method Formula, General and Total
specific_department = stats_bulletin[
    stats_bulletin['Department'] == i
].reset_index(drop=True)

#Input text cell-by-cell
worksheet.write('A5', f"Value by allocation method")  # Table title
worksheet.write('B5', f"£ millions", currency_format) #legend
worksheet.write('A6', f"Allocation method", header)   # column heading
worksheet.write('B6', f"Total spend", header)         #column heading

In [ ]:
#Creation of Table, colours and cells depending on length of table
row = 6
text_color = text_light
numerical_color = numerical_light

if len(specific_department) > 0:
    if specific_department.at[0, 'Formula'] > 0:
        worksheet.write(row, 0, 'Formula', text_color)
        worksheet.write(
            row,
            1,
            int(round(specific_department.at[0, 'Formula'] / 1000000, 0)),
            numerical_color
        )
        row += 1

        text_color = text_dark  # if formula row then color changes for general
        numerical_color = numerical_dark  # if formula row then color changes for general

    if specific_department.at[0, 'General'] > 0:
        worksheet.write(row, 0, 'General', text_color)
        worksheet.write(
            row,
            1,
            int(round(specific_department.at[0, 'General'] / 1000000, 0)),
            numerical_color
        )
        row += 1
		if row == 8:
			text_color = text_light
			numerical_color = numerical_light  # if table includes both formula and general then total row same color as formula
		else:
			text_color = text_dark
			numerical_color = numerical_dark  # if table includes one of formula and general then total row the other color

		worksheet.write(row, 0, 'Total', text_color)
		worksheet.write(
			row,
			1,
			int(round(specific_department.at[0, 'Total'] / 1000000, 0)),
			numerical_color
		)

		#Table of schemes listed in the top ten graphs
		department_schemes = largest_schemes[
			largest_schemes['Department'] == i
		].reset_index(drop=True)

		department_schemes = department_schemes[[
			'Scheme Name',
			'Purpose and objectives',
			'Allocation method',
			'Value'
		]].sort_values(
			by=['Value'],
			ascending=False
		).head(10).reset_index(drop=True)

		#Set column width in sheet
		worksheet.set_column('A:A', 25)
		worksheet.set_column('B:B', 20)
		worksheet.set_column('C:C', 17)
		worksheet.set_column('D:E', 34)
		worksheet.set_column('G:G', 20)

		#Input text cell-by-cell

		#Input headings of table
		row = 5  # the row the table starts in the excel
		column = 3  # the column the table starts in the excel

		if len(department_schemes) > 0:  #only build table if 1 or more schemes to list
			worksheet.write('D5', "Largest schemes")
			worksheet.write('G5', f"£ millions")

			for k in department_schemes.columns:  #loop through each heading
				worksheet.write(row, column, k, header)  #write each heading in cell
				column += 1

			#Input data of the tables
			row += 1  # row below headings

			department_schemes = department_schemes.fillna('')  # fill empty cells with blank string

			for l in range(len(department_schemes)):  # iterate over length of dataframe
				for m in range(department_schemes.shape[1]):  # iterate over width of dataframe
					if (l % 2) == 0:  # every second row dark
						if m == 3:  # if last column use numerical formatting
							worksheet.write(
								row,
								m+3,
								int(round(float(department_schemes.values[l][m]) / 1000000, 0)),
								numerical_light
							)
						else:
							worksheet.write(
								row,
								m+3,
								department_schemes.values[l][m],
								text_light
							)
					else:  # every other cell light
						if m == 3:  # if last column use numerical formatting
							worksheet.write(
								row,
								m+3,
								int(round(float(department_schemes.values[l][m]) / 1000000, 0)),
								numerical_dark
							)
						else:
							worksheet.write(
								row,
								m+3,
								department_schemes.values[l][m],
								text_dark
							)
				row += 1

		#Input text cell-by-cell
		worksheet.write(
			'A1',
			f"This sheet contains the figures for your department that we are going to publish"
		)
		worksheet.write(
			'A2',
			f"Please let us know if any of these figures are incorrect"
		)

		number_of_schemes = len(department_schemes)

		if number_of_schemes > 0:
			worksheet.write(
				'A3',
				f"We will also be highlighting {number_of_schemes} of your schemes as they are in the top 10 largest schemes for their allocation method - please check these and reword the name or purpose if necessary"
			)
    ####################################################################################################
    #Creating third sheet - 'Schemes to publish'
    
    #List of all schemes relating to the department
    signoff_scheme = scheme_register[
        scheme_register['Funder: Organisation Name'] ==
        department_acronym.at[a, 'Funder: Organisation Name']
    ]
    
    # Text has already travelled through the UTF-8 pipeline; normalise once more
    # immediately before sign-off export and fail closed if corruption remains.
    signoff_scheme = clean_frame(signoff_scheme).fillna('')
    scheme_encoding_issues = find_mojibake(signoff_scheme)
    if not scheme_encoding_issues.empty:
        print(scheme_encoding_issues.head(50))
        raise ValueError(
            f"Encoding QA failed for {i}: {len(scheme_encoding_issues)} suspect scheme cells. "
            "No sign-off workbook was produced for this department."
        )
    
    signoff_scheme.to_excel(
        writer,
        sheet_name='Schemes to publish',
        index=False
    )
    
    
    #List of all schemes relating to the department
    signoff_award = award_register[
        award_register['Funding Org:Name'] ==
        department_acronym.at[a, 'Funder: Organisation Name']
    ]
    
    signoff_award = clean_frame(signoff_award).fillna('')
    award_encoding_issues = find_mojibake(signoff_award)
    if not award_encoding_issues.empty:
        print(award_encoding_issues.head(50))
        raise ValueError(
            f"Encoding QA failed for {i}: {len(award_encoding_issues)} suspect award cells. "
            "No sign-off workbook was produced for this department."
        )
    
    signoff_award.to_excel(
        writer,
        sheet_name='Awards to publish',
        index=False
    )
    
    print(i)
    writer.save()
    writer.close()

In [ ]:
!pip install odfpy

In [ ]:
import pandas as pd

url = "https://assets.publishing.service.gov.uk/media/67e135ab64220b68ed6a6fdb/2025-03-24_Government_grants_statistical_tables_2023_to_2024.ods"

data = pd.read_excel(
    url,
    engine="odf"
)

data.head()

In [ ]:
#Import Previous publication data from assets.publishing.service.gov.uk

url = "https://assets.publishing.service.gov.uk/media/67e51552621ba30ed9776bf/2024-03-19_Government_grants_statistical_tables_2022_to_2023.ods"

YoY_Grants = pd.read_excel(
    url,
    engine='odf',
    sheet_name='Table_1',
    index_col=None
)

YoY_Grants_header = YoY_Grants.iloc[4]
YoY_Grants = YoY_Grants[5:]
YoY_Grants.columns = YoY_Grants_header

YoY_Grants.head()

In [ ]:
#YoY_Grants = YoY_Grants.style.set_table_styles([
#    {'selector': 'td',
#     'props': [('border-style', 'solid'), ('border-width', '1px')]}
#])

In [ ]:
YoY_Grants.to_excel(
    'final_Table_1.xlsx',
    sheet_name='Table_1',
    index=False
)

In [ ]:
with pd.ExcelWriter('final_Table_1.xlsx', engine='xlsxwriter') as writer:
    YoY_Grants.to_excel(
        writer,
        sheet_name='Sheet1',
        startrow=5,
        startcol=0,
        index=False
    )

    workbook = writer.book
    worksheet = writer.sheets['Sheet1']

    border_format = workbook.add_format({'border': 1})
    worksheet.set_column('A:A', 15)

    # Formats
    left_border_fmt = workbook.add_format({'left': 2})
    right_border_fmt = workbook.add_format({'right': 2})

In [ ]:
import pandas as pd

url = "https://assets.publishing.service.gov.uk/media/67e135ab64220b68ed6a6fdb/2025-03-24_Government_grants_statistical_tables_2023_to_2024.ods"
data = pd.read_excel(url, engine="odf")

data.head()

In [ ]:
#Import Previous publication data from assets.publishing.service.gov.uk

url = "https://assets.publishing.service.gov.uk/media/67e51552621ba30ed9776bf/2024-03-19_Government_grants_statistical_tables_2022_to_2023.ods"

YoY_Grants = pd.read_excel(
    url,
    engine='odf',
    sheet_name='Table_1',
    index_col=None
)

YoY_Grants_header = YoY_Grants.iloc[4]
YoY_Grants = YoY_Grants[5:]
YoY_Grants.columns = YoY_Grants_header

YoY_Grants.head()

In [ ]:
#YoY_Grants = YoY_Grants.style.set_table_styles([
#    {'selector': 'td',
#     'props': [('border-style','solid'),('border-width','1px')]}
#])

In [ ]:
YoY_Grants.to_excel(
    'final_Table_1.xlsx',
    sheet_name='Table_1',
    index=False
)

In [ ]:
with pd.ExcelWriter('final_Table_1.xlsx', engine='xlsxwriter') as writer:
    YoY_Grants.to_excel(
        writer,
        sheet_name='Sheet1',
        startrow=5,
        startcol=0,
        index=False
    )

    workbook = writer.book
    worksheet = writer.sheets['Sheet1']

    border_format = workbook.add_format({'border': 1})
    worksheet.set_column('A:A', 15)

    # Formats
    left_border_fmt = workbook.add_format({'left': 2})
    right_border_fmt = workbook.add_format({'right': 2})
    top_border_fmt = workbook.add_format({'top': 2, 'bottom': 2})
    bottom_border_fmt = workbook.add_format({'bottom': 2, 'top': 2})

    # Column width
    worksheet.set_column('A:A', 15)

    # Conditional formats (no_errors type)
    worksheet.conditional_format(
        'B6:F6',
        {'type': 'no_errors', 'format': top_border_fmt}
    )

    worksheet.conditional_format(
        'B6:B26',
        {'type': 'no_errors', 'format': left_border_fmt}
    )

    worksheet.conditional_format(
        'F6:F25',
        {'type': 'no_errors', 'format': right_border_fmt}
    )

    worksheet.conditional_format(
        'G6:K6',
        {'type': 'no_errors', 'format': top_border_fmt}
    )

    worksheet.conditional_format(
        'G26:K26',
        {'type': 'no_errors', 'format': bottom_border_fmt}
    )

    worksheet.conditional_format(
        'G6:G26',
        {'type': 'no_errors', 'format': left_border_fmt}
    )

    worksheet.conditional_format(
        'K6:K26',
        {'type': 'no_errors', 'format': right_border_fmt}
    )

    worksheet.conditional_format(
        'L6:O6',
        {'type': 'no_errors', 'format': top_border_fmt}
    )

    worksheet.conditional_format(
        'L26:O26',
        {'type': 'no_errors', 'format': bottom_border_fmt}
    )

    worksheet.conditional_format(
        'L6:L26',
        {'type': 'no_errors', 'format': left_border_fmt}
    )

    worksheet.conditional_format(
        'P6:P25',
        {'type': 'no_errors', 'format': right_border_fmt}
    )

    worksheet.conditional_format(
        'P26:P26',
        {'type': 'no_errors', 'format': bottom_border_fmt}
    )

print("Created File")

In [ ]:
import os

In [ ]:
os.getcwd()

In [ ]:
YoY_Grants.columns